In [ ]:
import pandas as pd

# Load dataset
file_path = "/kaggle/input/datasets/bassam165/earthquakegemfaults/earthquakes_1900_2026_combined.csv"
df = pd.read_csv(file_path)

# Total rows and columns
print("Shape (rows, columns):", df.shape)

# Column names
print("\nColumn Names:")
print(df.columns.tolist())

# Optional: Preview first few rows
df.head()

In [ ]:
!pip install -q torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu118

!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.2.0+cu118.html
!pip install -q torch-geometric

!pip install -q rtree pyproj statsmodels

# step 1

In [ ]:
# STEP 1: Data Loading & Preprocessing
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

OUT_DIR = "/kaggle/working/"
os.makedirs(OUT_DIR, exist_ok=True)

EQ_PATH    = "/kaggle/input/datasets/bassam165/earthquakegemfaults/earthquakes_1900_2026_combined.csv"
PB_PATH    = "/kaggle/input/datasets/bassam165/earthquakegemfaults/pb2002_boundaries.csv"
PLATE_PATH = "/kaggle/input/datasets/bassam165/earthquakegemfaults/pb2002_plates.csv"
GPKG_PATH  = "/kaggle/input/datasets/bassam165/earthquakegemfaults/gem_active_faults_harmonized.gpkg"

# ── 1. Load & Filter ─────────────────────────────────────────
print("Loading earthquake catalog...")
eq = pd.read_csv(EQ_PATH, low_memory=False)
eq['time'] = pd.to_datetime(eq['time'], utc=True, errors='coerce')
eq = eq.dropna(subset=['time','latitude','longitude','mag'])
eq = eq[eq['type'] == 'earthquake'].copy()
eq = eq[eq['mag'] >= 2.5].copy()
eq = eq.sort_values('time').reset_index(drop=True)
eq['depth'] = eq['depth'].fillna(eq['depth'].median())
print(f"  Total earthquakes loaded : {len(eq):,}")

# ── 2. Geophysical Layers ────────────────────────────────────
print("Loading geophysical layers...")
pb  = pd.read_csv(PB_PATH).dropna(subset=['latitude','longitude'])
flt = gpd.read_file(GPKG_PATH)
print(f"  Plate boundaries: {len(pb):,} | Faults: {len(flt):,}")

# ── 3. Fault Distances ───────────────────────────────────────
print("Computing fault distances...")
flt_valid       = flt[flt.geometry.notna()].copy()
flt_valid['cx'] = flt_valid.geometry.centroid.x
flt_valid['cy'] = flt_valid.geometry.centroid.y
flt_coords_rad  = np.radians(flt_valid[['cy','cx']].values.astype(np.float64))
eq_coords_rad   = np.radians(eq[['latitude','longitude']].values.astype(np.float64))
tree_flt        = cKDTree(flt_coords_rad)
dist_r, _       = tree_flt.query(eq_coords_rad, k=1, workers=-1)
eq['dist_fault_km'] = dist_r * 6371.0
print("  Fault distances done.")

# ── 4. Plate Boundary Distances ──────────────────────────────
print("Computing plate boundary distances...")
pb_coords_rad = np.radians(pb[['latitude','longitude']].values.astype(np.float64))
tree_pb       = cKDTree(pb_coords_rad)
dist_pb_r, _  = tree_pb.query(eq_coords_rad, k=1, workers=-1)
eq['dist_pb_km'] = dist_pb_r * 6371.0
print("  Plate boundary distances done.")

# ── 5. ETAS Features ─────────────────────────────────────────
print("Computing ETAS features...")
eq['timestamp_s'] = eq['time'].astype(np.int64) / 1e9
ts_arr  = eq['timestamp_s'].values.astype(np.float64)
mag_arr = eq['mag'].values.astype(np.float64)
N       = len(eq)

WINDOW_SEC = 72 * 3600
C_OMORI    = 0.01
P_OMORI    = 1.1
K_PROD     = 0.1

etas_lambda  = np.zeros(N, dtype=np.float32)
etas_mu      = np.zeros(N, dtype=np.float32)
etas_n_prior = np.zeros(N, dtype=np.float32)
etas_max_mag = np.zeros(N, dtype=np.float32)
etas_energy  = np.zeros(N, dtype=np.float32)

for i in range(N):
    lo = np.searchsorted(ts_arr, ts_arr[i] - WINDOW_SEC, side='left')
    hi = i
    if hi <= lo:
        continue
    prior_mag       = mag_arr[lo:hi]
    dt_days         = np.clip((ts_arr[i] - ts_arr[lo:hi]) / 86400.0, 1e-6, None)
    triggered       = K_PROD * np.power(10, 0.5*prior_mag) / np.power(dt_days + C_OMORI, P_OMORI)
    etas_lambda[i]  = triggered.sum()
    etas_mu[i]      = len(prior_mag) / 72.0
    etas_n_prior[i] = len(prior_mag)
    etas_max_mag[i] = prior_mag.max()
    etas_energy[i]  = np.sum(np.power(10, 1.5 * prior_mag))
    if i % 50000 == 0:
        print(f"  ETAS: {i:,}/{N:,}", end='\r')

eq['etas_lambda']  = etas_lambda
eq['etas_mu']      = etas_mu
eq['etas_n_prior'] = etas_n_prior
eq['etas_max_mag'] = etas_max_mag
eq['etas_energy']  = etas_energy
print(f"\n  ETAS features done.")

# ── 6. Temporal Features ─────────────────────────────────────
print("Computing temporal features...")
eq['hour_sin']  = np.sin(2 * np.pi * eq['time'].dt.hour / 24)
eq['hour_cos']  = np.cos(2 * np.pi * eq['time'].dt.hour / 24)
eq['doy_sin']   = np.sin(2 * np.pi * eq['time'].dt.dayofyear / 365)
eq['doy_cos']   = np.cos(2 * np.pi * eq['time'].dt.dayofyear / 365)
eq['dt_prev_s'] = eq['timestamp_s'].diff().fillna(0).clip(lower=0)

# ── 7. Feature Matrix & Scaling ──────────────────────────────
FEATURE_COLS = [
    'latitude', 'longitude', 'depth', 'mag',
    'dist_fault_km', 'dist_pb_km',
    'etas_lambda', 'etas_mu', 'etas_n_prior', 'etas_max_mag', 'etas_energy',
    'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'dt_prev_s'
]
scaler   = StandardScaler()
eq_scaled = eq.copy()
eq_scaled[FEATURE_COLS] = scaler.fit_transform(eq[FEATURE_COLS])
print(f"  Feature matrix : {eq_scaled[FEATURE_COLS].shape}")

# ── 8. Raw Array ─────────────────────────────────────────────
eq_arr_raw = eq[['timestamp_s','latitude','longitude',
                 'mag','depth']].fillna(0).values.astype(np.float64)
print(f"  Raw event array : {eq_arr_raw.shape}")

# ── 9. Label Builder (searchsorted + spatial KDTree) ─────────
print("Building labels...")

HORIZONS_SEC = {
    '1h':  1  * 3600,
    '6h':  6  * 3600,
    '12h': 12 * 3600,
    '24h': 24 * 3600,
    '72h': 72 * 3600,
}
RADIUS_KM = 150.0
R_EARTH   = 6371.0

def build_labels_kdtree(eq_arr_raw, horizons, radius_km):
    """
    For each event i:
      1. searchsorted finds future events in time window → tiny slice
      2. cKDTree on that slice finds spatial neighbours within radius_km
    No large matrix ever allocated — O(N * W) where W = avg window size.
    """
    N   = len(eq_arr_raw)
    ts  = eq_arr_raw[:, 0]
    lat = eq_arr_raw[:, 1]
    lon = eq_arr_raw[:, 2]
    mag = eq_arr_raw[:, 3]

    # Convert all coords to radians once
    coords_rad = np.radians(np.stack([lat, lon], axis=1))

    # Radius in radians for cKDTree (chord approximation)
    radius_rad = radius_km / R_EARTH

    labels = {}
    max_h  = max(horizons.values())

    for h_name, h_sec in horizons.items():
        occ  = np.zeros(N, dtype=np.float32)
        magn = np.zeros(N, dtype=np.float32)
        lats = np.zeros(N, dtype=np.float32)
        lons = np.zeros(N, dtype=np.float32)

        for i in range(N):
            # Time slice using searchsorted — O(log N)
            lo = np.searchsorted(ts, ts[i] + 1,       side='left')
            hi = np.searchsorted(ts, ts[i] + h_sec,   side='right')
            if hi <= lo:
                continue

            # Candidate future events in time window
            cand_coords = coords_rad[lo:hi]
            cand_mag    = mag[lo:hi]
            cand_lat    = lat[lo:hi]
            cand_lon    = lon[lo:hi]

            # Spatial filter with cKDTree on candidate slice
            if len(cand_coords) == 0:
                continue

            tree      = cKDTree(cand_coords)
            idx_near  = tree.query_ball_point(coords_rad[i], r=radius_rad)

            if len(idx_near) == 0:
                continue

            occ[i]  = 1.0
            magn[i] = cand_mag[idx_near].max()
            lats[i] = cand_lat[idx_near].mean()
            lons[i] = cand_lon[idx_near].mean()

            if i % 50000 == 0:
                print(f"  {h_name}: {i:,}/{N:,}", end='\r')

        labels[h_name] = {
            'occurrence': occ, 'magnitude': magn,
            'lat': lats,       'lon': lons
        }
        print(f"  {h_name:>4s}: occurrence={occ.mean():.3f}  "
              f"positives={int(occ.sum()):,}    ")

    return labels

labels = build_labels_kdtree(eq_arr_raw, HORIZONS_SEC, RADIUS_KM)

# ── 10. Save ──────────────────────────────────────────────────
print("\nSaving outputs...")
eq_scaled.to_csv(f"{OUT_DIR}eq_features.csv", index=False)
np.save(f"{OUT_DIR}eq_arr_raw.npy", eq_arr_raw)

with open(f"{OUT_DIR}labels.pkl", 'wb') as f:
    pickle.dump(labels, f)
with open(f"{OUT_DIR}scaler.pkl", 'wb') as f:
    pickle.dump(scaler, f)

print(f"\nStep 1 complete.")
print(f"  Dataset size   : {len(eq):,} events")
print(f"  Feature matrix : {eq_scaled[FEATURE_COLS].shape}")
print(f"  Horizons       : {list(labels.keys())}")
print(f"  Saved to       : {OUT_DIR}")

# Step 2

In [ ]:
# STEP 2: Graph Construction (Chunk-Only, No Merge)

import os
import gc
import numpy as np
import pandas as pd
import pickle
import torch
from torch_geometric.data import Data
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

OUT_DIR    = "/kaggle/working/"
CHUNKS_DIR = f"{OUT_DIR}graph_chunks/"
os.makedirs(CHUNKS_DIR, exist_ok=True)

# ── Load ─────────────────────────────────────────────────────
print("Loading preprocessed data...")
eq         = pd.read_csv(f"{OUT_DIR}eq_features.csv")
eq_arr_raw = np.load(f"{OUT_DIR}eq_arr_raw.npy")

with open(f"{OUT_DIR}labels.pkl", 'rb') as f:
    labels = pickle.load(f)

FEATURE_COLS = [
    'latitude', 'longitude', 'depth', 'mag',
    'dist_fault_km', 'dist_pb_km',
    'etas_lambda', 'etas_mu', 'etas_n_prior', 'etas_max_mag', 'etas_energy',
    'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'dt_prev_s'
]
X_all = eq[FEATURE_COLS].values.astype(np.float32)

HORIZONS      = ['1h', '6h', '12h', '24h', '72h']
INPUT_WINDOWS = {'w24': 24*3600, 'w48': 48*3600, 'w72': 72*3600}

K_NEIGHBORS = 6
MAX_EDGE_KM = 300.0
ETAS_THRESH = 0.05
MIN_EVENTS  = 5
MAX_EVENTS  = 80
STEP        = 20
CHUNK_SIZE  = 5000
N_JOBS      = 2

N              = len(eq_arr_raw)
timestamps     = eq_arr_raw[:, 0]
anchor_indices = list(range(MIN_EVENTS, N, STEP))

print(f"Total events     : {N:,}")
print(f"Anchor step      : every {STEP}th event")
print(f"Total anchors    : {len(anchor_indices):,}")
print(f"Max events/graph : {MAX_EVENTS}")
print(f"Chunk size       : {CHUNK_SIZE}")
print(f"n_jobs           : {N_JOBS}")

# ── Check disk space ─────────────────────────────────────────
import shutil
total, used, free = shutil.disk_usage(OUT_DIR)
print(f"Disk free        : {free/1e9:.1f} GB")

# ── Vectorised Haversine ──────────────────────────────────────
def haversine_matrix_vec(lats, lons):
    R     = 6371.0
    lat_r = np.radians(lats)
    lon_r = np.radians(lons)
    dlat  = lat_r[:, None] - lat_r[None, :]
    dlon  = lon_r[:, None] - lon_r[None, :]
    a     = (np.sin(dlat/2)**2 +
             np.cos(lat_r[:, None]) * np.cos(lat_r[None, :]) *
             np.sin(dlon/2)**2)
    return R * 2 * np.arcsin(np.sqrt(a).clip(0, 1))

# ── Single Graph Builder ──────────────────────────────────────
def build_graph(event_indices, X_all, eq_arr_raw, labels):
    idx  = np.array(event_indices)
    n    = len(idx)
    feat = X_all[idx]
    raw  = eq_arr_raw[idx]
    lats = raw[:, 1]
    lons = raw[:, 2]
    ts   = raw[:, 0]
    mags = raw[:, 3]

    dist_mat   = haversine_matrix_vec(lats, lons)
    coords_rad = np.stack([np.radians(lats), np.radians(lons)], axis=1)
    tree       = cKDTree(coords_rad)
    kk         = min(K_NEIGHBORS + 1, n)
    _, nbrs    = tree.query(coords_rad, k=kk)

    src_list, dst_list, dist_list = [], [], []

    for i in range(n):
        for j_pos in range(1, kk):
            j = nbrs[i, j_pos]
            if dist_mat[i, j] <= MAX_EDGE_KM:
                src_list.append(i)
                dst_list.append(j)
                dist_list.append(dist_mat[i, j])

    C, P, Kp = 0.01, 1.1, 0.1
    dt_mat   = (ts[:, None] - ts[None, :]) / 86400.0
    valid    = dt_mat > 0
    with np.errstate(divide='ignore', invalid='ignore'):
        trig = np.where(
            valid,
            Kp * np.power(10, 0.5 * mags[None, :]) /
            np.power(np.where(valid, dt_mat + C, 1.0), P),
            0.0)
    e_src, e_dst = np.where(trig >= ETAS_THRESH)
    for s, d in zip(e_src.tolist(), e_dst.tolist()):
        src_list.append(int(s))
        dst_list.append(int(d))
        dist_list.append(float(dist_mat[s, d]))

    if len(src_list) == 0:
        for i in range(n):
            for j in range(n):
                if i != j:
                    src_list.append(i)
                    dst_list.append(j)
                    dist_list.append(0.0)

    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    etas_col   = FEATURE_COLS.index('etas_lambda')
    n_e        = len(src_list)

    ef_dist = np.array(dist_list, dtype=np.float32) / MAX_EDGE_KM
    ef_dt   = np.zeros(n_e, dtype=np.float32)
    ef_dmag = np.zeros(n_e, dtype=np.float32)
    ef_etas = np.zeros(n_e, dtype=np.float32)

    for e, (s, d) in enumerate(zip(src_list, dst_list)):
        ef_dt[e]   = np.log1p(abs(ts[d] - ts[s])) / np.log1p(72 * 3600)
        ef_dmag[e] = abs(mags[s] - mags[d]) / 10.0
        ef_etas[e] = float(feat[s, etas_col])

    edge_attr = torch.tensor(
        np.stack([ef_dist, ef_dt, ef_dmag, ef_etas], axis=1),
        dtype=torch.float32)
    x      = torch.tensor(feat, dtype=torch.float32)
    anchor = idx[-1]
    data   = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    for h in HORIZONS:
        setattr(data, f'{h}_occ',
                torch.tensor([labels[h]['occurrence'][anchor]], dtype=torch.float32))
        setattr(data, f'{h}_mag',
                torch.tensor([labels[h]['magnitude'][anchor]],  dtype=torch.float32))
        setattr(data, f'{h}_lat',
                torch.tensor([labels[h]['lat'][anchor]],        dtype=torch.float32))
        setattr(data, f'{h}_lon',
                torch.tensor([labels[h]['lon'][anchor]],        dtype=torch.float32))

    data.num_nodes  = n
    data.anchor_idx = torch.tensor([anchor], dtype=torch.long)
    return data

# ── Per-window job ────────────────────────────────────────────
def build_one_graph(i, timestamps, X_all, eq_arr_raw, labels, w_sec):
    t_anchor = timestamps[i]
    mask     = ((timestamps[:i+1] >= t_anchor - w_sec) &
                (timestamps[:i+1] <= t_anchor))
    win_idx  = np.where(mask)[0]
    if len(win_idx) < MIN_EVENTS:
        return None
    if len(win_idx) > MAX_EVENTS:
        win_idx = win_idx[-MAX_EVENTS:]
    try:
        return build_graph(win_idx, X_all, eq_arr_raw, labels)
    except Exception:
        return None

# ── Chunked build + save ──────────────────────────────────────
def build_window_chunked(w_name, w_sec, anchor_indices,
                         timestamps, X_all, eq_arr_raw, labels):
    # Skip if all chunks already exist
    existing = sorted([
        f for f in os.listdir(CHUNKS_DIR)
        if f.startswith(w_name) and f.endswith('.pt')
    ])
    anchor_chunks = [
        anchor_indices[s:s+CHUNK_SIZE]
        for s in range(0, len(anchor_indices), CHUNK_SIZE)
    ]
    if len(existing) == len(anchor_chunks):
        total = sum(
            len(torch.load(f"{CHUNKS_DIR}{f}", weights_only=False))
            for f in existing)
        print(f"  {w_name}: already built ({total:,} graphs) — skipping")
        return total

    total = sum(
        len(torch.load(f"{CHUNKS_DIR}{f}", weights_only=False))
        for f in existing)
    start_chunk = len(existing)

    print(f"\n  {w_name}: {len(anchor_indices):,} anchors → "
          f"{len(anchor_chunks)} chunks  "
          f"(resuming from chunk {start_chunk})")

    for ci in range(start_chunk, len(anchor_chunks)):
        chunk_idx = anchor_chunks[ci]
        results   = Parallel(n_jobs=N_JOBS, backend='loky', verbose=0)(
            delayed(build_one_graph)(
                i, timestamps, X_all, eq_arr_raw, labels, w_sec)
            for i in chunk_idx
        )
        graphs_chunk = [g for g in results if g is not None]
        if len(graphs_chunk) > 0:
            chunk_path = f"{CHUNKS_DIR}{w_name}_chunk{ci:04d}.pt"
            torch.save(graphs_chunk, chunk_path)
            total += len(graphs_chunk)

        del results, graphs_chunk
        gc.collect()
        print(f"    chunk {ci+1:>4}/{len(anchor_chunks)}  "
              f"saved {total:,} graphs so far", end='\r')

    print(f"\n  {w_name}: {total:,} graphs in "
          f"{len(anchor_chunks)} chunk files")
    return total

# ── Build all windows ─────────────────────────────────────────
# Delete the large merged file if it exists to free disk space
for w in INPUT_WINDOWS:
    merged_path = f"{OUT_DIR}graphs_{w}.pt"
    if os.path.exists(merged_path):
        os.remove(merged_path)
        print(f"Removed old merged file: graphs_{w}.pt")

window_counts = {}
for w_name, w_sec in INPUT_WINDOWS.items():
    print(f"\nProcessing {w_name}...")
    count = build_window_chunked(
        w_name, w_sec, anchor_indices,
        timestamps, X_all, eq_arr_raw, labels)
    window_counts[w_name] = count
    gc.collect()

print(f"\nWindow counts : {window_counts}")

# ── Compute aligned size ──────────────────────────────────────
N_common = min(window_counts.values())
print(f"Aligned size  : {N_common:,} graphs per window")

# ── Build chunk index for each window ────────────────────────
# Instead of one big file, store a list of chunk paths + offsets
# so DataLoader can stream from disk

def build_chunk_index(w_name, n_common):
    """
    Returns list of (chunk_path, start, end) tuples
    covering exactly n_common graphs in order.
    """
    chunk_files = sorted([
        f"{CHUNKS_DIR}{f}"
        for f in os.listdir(CHUNKS_DIR)
        if f.startswith(w_name) and f.endswith('.pt')
    ])
    index  = []
    total  = 0
    for cf in chunk_files:
        part_len = len(torch.load(cf, weights_only=False))
        if total >= n_common:
            break
        take   = min(part_len, n_common - total)
        index.append({'path': cf, 'start': 0, 'end': take})
        total += take
    return index

chunk_index = {}
for w_name in INPUT_WINDOWS:
    chunk_index[w_name] = build_chunk_index(w_name, N_common)
    n_indexed = sum(e['end'] - e['start'] for e in chunk_index[w_name])
    print(f"  {w_name}: {len(chunk_index[w_name])} chunk files "
          f"→ {n_indexed:,} graphs indexed")

# ── Save metadata ─────────────────────────────────────────────
# Load a sample graph for shape info
sample_chunk = torch.load(
    chunk_index['w24'][0]['path'], weights_only=False)
sample = sample_chunk[0]
del sample_chunk
gc.collect()

try:
    with open(f"{OUT_DIR}dataset_meta.pkl", 'rb') as f:
        meta = pickle.load(f)
except FileNotFoundError:
    meta = {}

meta['N_common']    = N_common
meta['n_features']  = sample.x.shape[1]
meta['n_edge_feat'] = sample.edge_attr.shape[1]
meta['horizons']    = HORIZONS
meta['step']        = STEP
meta['max_events']  = MAX_EVENTS
meta['chunk_index'] = chunk_index
meta['chunks_dir']  = CHUNKS_DIR

with open(f"{OUT_DIR}dataset_meta.pkl", 'wb') as f:
    pickle.dump(meta, f)

# ── Disk usage summary ────────────────────────────────────────
total_chunk_size = sum(
    os.path.getsize(f"{CHUNKS_DIR}{f}")
    for f in os.listdir(CHUNKS_DIR)
) / 1e9
_, _, free = shutil.disk_usage(OUT_DIR)
print(f"\nChunk storage  : {total_chunk_size:.1f} GB")
print(f"Disk remaining : {free/1e9:.1f} GB")

print(f"\nSample graph (w24):")
print(f"  Nodes      : {sample.num_nodes}")
print(f"  Edges      : {sample.edge_index.shape[1]}")
print(f"  Node feats : {sample.x.shape}")
print(f"  Edge feats : {sample.edge_attr.shape}")

print(f"\nStep 2 complete.")
print(f"  Graphs per window : {N_common:,}")
print(f"  Chunk files       : {CHUNKS_DIR}")
print(f"  Metadata saved    : dataset_meta.pkl")

# Step 3

In [ ]:
# STEP 3: Dataset & DataLoader (Chunk-Streaming)

import os
import gc
import pickle
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torch_geometric.data import Data, Batch
import warnings
warnings.filterwarnings('ignore')

OUT_DIR  = "/kaggle/working/"
HORIZONS = ['1h', '6h', '12h', '24h', '72h']

# ── Load Metadata ─────────────────────────────────────────────
with open(f"{OUT_DIR}dataset_meta.pkl", 'rb') as f:
    meta = pickle.load(f)

N_common    = meta['N_common']
chunk_index = meta['chunk_index']   # {w24: [{path, start, end},...], ...}
CHUNKS_DIR  = meta['chunks_dir']

print(f"N_common     : {N_common:,}")
print(f"Chunk files  : {sum(len(v) for v in chunk_index.values())} total")

# ── Build flat graph list from chunk index ────────────────────
# Load all chunks window by window, free after each window
# Store as flat list in memory — chunks are small enough per window

def load_graphs_from_chunks(w_name, chunk_index, n_common):
    """Stream all chunks for one window into a flat list."""
    graphs = []
    entries = chunk_index[w_name]
    for entry in entries:
        part = torch.load(entry['path'], weights_only=False)
        graphs.extend(part[entry['start']:entry['end']])
        del part
        gc.collect()
        if len(graphs) >= n_common:
            break
    return graphs[:n_common]

print("Loading w24 graphs...")
graphs_w24 = load_graphs_from_chunks('w24', chunk_index, N_common)
print(f"  w24: {len(graphs_w24):,} graphs")

print("Loading w48 graphs...")
graphs_w48 = load_graphs_from_chunks('w48', chunk_index, N_common)
print(f"  w48: {len(graphs_w48):,} graphs")

print("Loading w72 graphs...")
graphs_w72 = load_graphs_from_chunks('w72', chunk_index, N_common)
print(f"  w72: {len(graphs_w72):,} graphs")

# ── Dataset ───────────────────────────────────────────────────
class AfterShockDataset(Dataset):
    """
    Each sample returns three parallel graphs (w24, w48, w72)
    for the same anchor event plus a flat label vector [20].

    Label layout:
      [1h_occ, 1h_mag, 1h_lat, 1h_lon,
       6h_occ, 6h_mag, 6h_lat, 6h_lon,
       12h_occ,12h_mag,12h_lat,12h_lon,
       24h_occ,24h_mag,24h_lat,24h_lon,
       72h_occ,72h_mag,72h_lat,72h_lon]
    """
    def __init__(self, g24, g48, g72, horizons):
        self.g24      = g24
        self.g48      = g48
        self.g72      = g72
        self.horizons = horizons

    def __len__(self):
        return len(self.g24)

    def __getitem__(self, idx):
        g24 = self.g24[idx]
        g48 = self.g48[idx]
        g72 = self.g72[idx]
        parts = []
        for h in self.horizons:
            parts += [
                getattr(g24, f'{h}_occ'),
                getattr(g24, f'{h}_mag'),
                getattr(g24, f'{h}_lat'),
                getattr(g24, f'{h}_lon'),
            ]
        labels = torch.cat(parts, dim=0)   # [20]
        return g24, g48, g72, labels

# ── Chronological Train / Val / Test Split ────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

n_train = int(N_common * TRAIN_RATIO)
n_val   = int(N_common * VAL_RATIO)
n_test  = N_common - n_train - n_val

train_idx = list(range(0,               n_train))
val_idx   = list(range(n_train,         n_train + n_val))
test_idx  = list(range(n_train + n_val, N_common))

print(f"\nSplit  →  train: {len(train_idx):,}  "
      f"val: {len(val_idx):,}  test: {len(test_idx):,}")

full_dataset  = AfterShockDataset(graphs_w24, graphs_w48, graphs_w72, HORIZONS)
train_dataset = Subset(full_dataset, train_idx)
val_dataset   = Subset(full_dataset, val_idx)
test_dataset  = Subset(full_dataset, test_idx)

# ── Collate Function ──────────────────────────────────────────
def collate_fn(batch):
    g24_list, g48_list, g72_list, label_list = zip(*batch)
    batch_g24 = Batch.from_data_list(list(g24_list))
    batch_g48 = Batch.from_data_list(list(g48_list))
    batch_g72 = Batch.from_data_list(list(g72_list))
    labels    = torch.stack(label_list, dim=0)   # [B, 20]
    return batch_g24, batch_g48, batch_g72, labels

# ── DataLoaders ───────────────────────────────────────────────
BATCH_SIZE  = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

# ── Label Statistics for Loss Weighting ──────────────────────
print("\nComputing label statistics...")
all_labels = torch.stack(
    [full_dataset[i][3] for i in range(N_common)], dim=0)   # [N, 20]

pos_weights = {}
label_stds  = {}

print(f"\n{'Horizon':<8} {'Positives':>10} {'Negatives':>10} "
      f"{'PosWeight':>10} {'MagStd':>8}")
print("-" * 52)

for h_idx, h in enumerate(HORIZONS):
    base    = h_idx * 4
    occ_col = all_labels[:, base]
    mag_col = all_labels[:, base + 1]
    lat_col = all_labels[:, base + 2]
    lon_col = all_labels[:, base + 3]

    n_pos = occ_col.sum().item()
    n_neg = N_common - n_pos
    pw    = n_neg / max(n_pos, 1)
    pos_weights[h] = torch.tensor([pw], dtype=torch.float32)

    mask    = occ_col > 0
    mag_std = mag_col[mask].std().item() if mask.sum() > 1 else 1.0
    lat_std = lat_col[mask].std().item() if mask.sum() > 1 else 1.0
    lon_std = lon_col[mask].std().item() if mask.sum() > 1 else 1.0
    label_stds[h] = {'mag': mag_std, 'lat': lat_std, 'lon': lon_std}

    print(f"{h:<8} {int(n_pos):>10,} {int(n_neg):>10,} "
          f"{pw:>10.2f} {mag_std:>8.4f}")

# ── Sanity Check ─────────────────────────────────────────────
print("\nSanity check — one training batch:")
bg24, bg48, bg72, lbl = next(iter(train_loader))
print(f"  g24 nodes     : {bg24.x.shape}")
print(f"  g24 edges     : {bg24.edge_index.shape}")
print(f"  g24 edge_attr : {bg24.edge_attr.shape}")
print(f"  g48 nodes     : {bg48.x.shape}")
print(f"  g72 nodes     : {bg72.x.shape}")
print(f"  labels        : {lbl.shape}  (B × 20)")

# ── Save Updated Metadata ─────────────────────────────────────
meta['n_train']     = n_train
meta['n_val']       = n_val
meta['n_test']      = n_test
meta['batch_size']  = BATCH_SIZE
meta['pos_weights'] = pos_weights
meta['label_stds']  = label_stds

with open(f"{OUT_DIR}dataset_meta.pkl", 'wb') as f:
    pickle.dump(meta, f)

print(f"\nStep 3 complete.")
print(f"  Train : {n_train:,}")
print(f"  Val   : {n_val:,}")
print(f"  Test  : {n_test:,}")
print(f"  Saved : dataset_meta.pkl")

# Step 4

In [ ]:
# STEP 4: Model Architecture

import os
import gc
import math
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from torch_geometric.data import Batch
import warnings
warnings.filterwarnings('ignore')

OUT_DIR  = "/kaggle/working/"
HORIZONS = ['1h', '6h', '12h', '24h', '72h']

with open(f"{OUT_DIR}dataset_meta.pkl", 'rb') as f:
    meta = pickle.load(f)

N_NODE_FEAT = meta['n_features']
N_EDGE_FEAT = meta['n_edge_feat']
chunk_index = meta['chunk_index']
CHUNKS_DIR  = meta['chunks_dir']

# ── Detect working device ─────────────────────────────────────
# Test if CUDA kernels are actually functional on this GPU
def get_device():
    if not torch.cuda.is_available():
        return torch.device('cpu')
    try:
        t = torch.zeros(2, 2).cuda()
        t = t @ t
        del t
        torch.cuda.empty_cache()
        return torch.device('cuda')
    except Exception as e:
        print(f"  CUDA test failed: {e}")
        print("  Falling back to CPU.")
        return torch.device('cpu')

DEVICE = get_device()
print(f"Device        : {DEVICE}")
print(f"Node features : {N_NODE_FEAT} | Edge features : {N_EDGE_FEAT}")

# ── Helper ────────────────────────────────────────────────────
def load_sample_graphs(w_name, chunk_index, n=2):
    entry  = chunk_index[w_name][0]
    part   = torch.load(entry['path'], weights_only=False)
    graphs = part[:n]
    del part
    gc.collect()
    return graphs

# 1. ETAS Prior Embedding
class ETASPriorEmbedding(nn.Module):
    def __init__(self, n_etas=5, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_etas, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, embed_dim),
            nn.GELU(),
        )
        self.etas_idx = [6, 7, 8, 9, 10]

    def forward(self, x):
        return self.net(x[:, self.etas_idx])

# 2. ST-GAT Block
class STGATBlock(nn.Module):
    def __init__(self, in_dim, out_dim, n_heads=4, edge_dim=4, dropout=0.1):
        super().__init__()
        assert out_dim % n_heads == 0
        self.gat = GATv2Conv(
            in_channels  = in_dim,
            out_channels = out_dim // n_heads,
            heads        = n_heads,
            edge_dim     = edge_dim,
            concat       = True,
            dropout      = dropout,
        )
        self.norm     = nn.LayerNorm(out_dim)
        self.dropout  = nn.Dropout(dropout)
        self.res_proj = (nn.Linear(in_dim, out_dim)
                         if in_dim != out_dim else nn.Identity())

    def forward(self, x, edge_index, edge_attr):
        residual = self.res_proj(x)
        out      = self.gat(x, edge_index, edge_attr)
        out      = self.dropout(out)
        return self.norm(out + residual)

# 3. Multi-Window GNN Encoder
class MultiWindowGNNEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim,
                 n_layers=3, n_heads=4, edge_dim=4,
                 etas_embed_dim=32, dropout=0.1):
        super().__init__()
        self.etas_embed  = ETASPriorEmbedding(n_etas=5, embed_dim=etas_embed_dim)
        self.input_proj  = nn.Linear(in_dim + etas_embed_dim, hidden_dim)
        self.gat_layers  = nn.ModuleList([
            STGATBlock(hidden_dim, hidden_dim, n_heads, edge_dim, dropout)
            for _ in range(n_layers)
        ])
        self.pool_proj   = nn.Linear(hidden_dim * 2, out_dim)
        self.window_fuse = nn.Sequential(
            nn.Linear(out_dim * 3, out_dim * 2),
            nn.LayerNorm(out_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim * 2, out_dim),
        )

    def encode_window(self, data):
        x          = data.x
        edge_index = data.edge_index
        edge_attr  = data.edge_attr
        batch      = data.batch
        etas_emb   = self.etas_embed(x)
        x          = torch.cat([x, etas_emb], dim=1)
        x          = self.input_proj(x)
        for layer in self.gat_layers:
            x = layer(x, edge_index, edge_attr)
        g_mean = global_mean_pool(x, batch)
        g_max  = global_max_pool(x, batch)
        return self.pool_proj(torch.cat([g_mean, g_max], dim=1))

    def forward(self, g24, g48, g72):
        e24 = self.encode_window(g24)
        e48 = self.encode_window(g48)
        e72 = self.encode_window(g72)
        return self.window_fuse(torch.cat([e24, e48, e72], dim=1))

# 4. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe           = torch.zeros(max_len, d_model)
        pos          = torch.arange(0, max_len).unsqueeze(1).float()
        div          = torch.exp(torch.arange(0, d_model, 2).float()
                                 * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

# 5. Temporal Transformer Encoder
class TemporalTransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads=8, n_layers=4,
                 ffn_dim=512, dropout=0.1, n_horizons=5):
        super().__init__()
        self.pos_enc        = PositionalEncoding(d_model, dropout=dropout)
        self.horizon_tokens = nn.Parameter(
            torch.randn(n_horizons, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = n_heads,
            dim_feedforward = ffn_dim,
            dropout         = dropout,
            activation      = 'gelu',
            batch_first     = True,
            norm_first      = True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers = n_layers,
            norm       = nn.LayerNorm(d_model),
        )

    def forward(self, gnn_emb):
        B        = gnn_emb.size(0)
        ctx      = gnn_emb.unsqueeze(1)
        h_tokens = self.horizon_tokens.unsqueeze(0).expand(B, -1, -1)
        seq      = torch.cat([ctx, h_tokens], dim=1)
        seq      = self.pos_enc(seq)
        out      = self.transformer(seq)
        return out[:, 1:, :]

# 6. Horizon Head
class HorizonHead(nn.Module):
    def __init__(self, d_model, hidden=128, dropout=0.1):
        super().__init__()
        def mlp(out_dim):
            return nn.Sequential(
                nn.Linear(d_model, hidden),
                nn.LayerNorm(hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, out_dim),
            )
        self.occ_head = mlp(1)
        self.mag_head = mlp(1)
        self.loc_head = mlp(2)

    def forward(self, h_emb):
        return self.occ_head(h_emb), self.mag_head(h_emb), self.loc_head(h_emb)


# 7. AfterShockNet
class AfterShockNet(nn.Module):
    def __init__(self,
                 n_node_feat=16, n_edge_feat=4,
                 gnn_hidden=128, gnn_out=256, gnn_layers=3,
                 gnn_heads=4,   etas_embed=32,
                 tf_d_model=256, tf_heads=8, tf_layers=4, tf_ffn=512,
                 head_hidden=128, dropout=0.1, n_horizons=5):
        super().__init__()
        self.gnn = MultiWindowGNNEncoder(
            in_dim         = n_node_feat,
            hidden_dim     = gnn_hidden,
            out_dim        = gnn_out,
            n_layers       = gnn_layers,
            n_heads        = gnn_heads,
            edge_dim       = n_edge_feat,
            etas_embed_dim = etas_embed,
            dropout        = dropout,
        )
        self.gnn_to_tf = (nn.Linear(gnn_out, tf_d_model)
                          if gnn_out != tf_d_model else nn.Identity())
        self.transformer = TemporalTransformerEncoder(
            d_model    = tf_d_model,
            n_heads    = tf_heads,
            n_layers   = tf_layers,
            ffn_dim    = tf_ffn,
            dropout    = dropout,
            n_horizons = n_horizons,
        )
        self.heads = nn.ModuleList([
            HorizonHead(tf_d_model, head_hidden, dropout)
            for _ in range(n_horizons)
        ])
        self.horizons   = HORIZONS
        self.n_horizons = n_horizons

    def forward(self, g24, g48, g72):
        gnn_emb = self.gnn(g24, g48, g72)
        tf_in   = self.gnn_to_tf(gnn_emb)
        h_embs  = self.transformer(tf_in)
        preds   = {}
        for i, h_name in enumerate(self.horizons):
            occ_logit, mag, loc = self.heads[i](h_embs[:, i, :])
            preds[h_name] = {
                'occ_logit': occ_logit,
                'mag':       mag,
                'loc':       loc,
            }
        return preds

# 8. Instantiate & Verify
MODEL_CFG = dict(
    n_node_feat = N_NODE_FEAT,
    n_edge_feat = N_EDGE_FEAT,
    gnn_hidden  = 128,
    gnn_out     = 256,
    gnn_layers  = 3,
    gnn_heads   = 4,
    etas_embed  = 32,
    tf_d_model  = 256,
    tf_heads    = 8,
    tf_layers   = 4,
    tf_ffn      = 512,
    head_hidden = 128,
    dropout     = 0.1,
    n_horizons  = 5,
)

model = AfterShockNet(**MODEL_CFG).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nAfterShockNet")
print(f"  Total params     : {total_params:,}")
print(f"  Trainable params : {trainable_params:,}")

# ── Forward Pass Sanity Check ─────────────────────────────────
print("\nRunning forward pass sanity check...")
s24 = load_sample_graphs('w24', chunk_index, n=2)
s48 = load_sample_graphs('w48', chunk_index, n=2)
s72 = load_sample_graphs('w72', chunk_index, n=2)

bg24 = Batch.from_data_list(s24).to(DEVICE)
bg48 = Batch.from_data_list(s48).to(DEVICE)
bg72 = Batch.from_data_list(s72).to(DEVICE)

# Run on CPU first to isolate CUDA kernel issues
print("  Testing on CPU first...")
model_cpu = AfterShockNet(**MODEL_CFG).cpu()
bg24_cpu  = Batch.from_data_list(s24).cpu()
bg48_cpu  = Batch.from_data_list(s48).cpu()
bg72_cpu  = Batch.from_data_list(s72).cpu()

model_cpu.eval()
with torch.no_grad():
    out_cpu = model_cpu(bg24_cpu, bg48_cpu, bg72_cpu)

print(f"  CPU forward pass OK")
for h in HORIZONS:
    print(f"    {h:>4s}  "
          f"occ:{out_cpu[h]['occ_logit'].shape}  "
          f"mag:{out_cpu[h]['mag'].shape}  "
          f"loc:{out_cpu[h]['loc'].shape}")

del model_cpu, bg24_cpu, bg48_cpu, bg72_cpu, out_cpu
gc.collect()

# Now test on DEVICE if it is CUDA
if DEVICE.type == 'cuda':
    print(f"\n  Testing on {DEVICE}...")
    try:
        model.eval()
        with torch.no_grad():
            out = model(bg24, bg48, bg72)
        print(f"  CUDA forward pass OK")
        for h in HORIZONS:
            print(f"    {h:>4s}  "
                  f"occ:{out[h]['occ_logit'].shape}  "
                  f"mag:{out[h]['mag'].shape}  "
                  f"loc:{out[h]['loc'].shape}")
        del out
        gc.collect()
    except Exception as e:
        print(f"  CUDA forward pass failed: {e}")
        print("  Training will use CPU. Consider restarting with GPU accelerator.")
        DEVICE = torch.device('cpu')
        model  = model.cpu()

del s24, s48, s72, bg24, bg48, bg72
gc.collect()

# ── Save ──────────────────────────────────────────────────────
torch.save(model.state_dict(), f"{OUT_DIR}aftershocknet_init.pt")

with open(f"{OUT_DIR}model_config.pkl", 'wb') as f:
    pickle.dump(MODEL_CFG, f)

# Save confirmed device for Step 5
meta['device'] = str(DEVICE)
with open(f"{OUT_DIR}dataset_meta.pkl", 'wb') as f:
    pickle.dump(meta, f)

print(f"\nStep 4 complete.")
print(f"  Device         : {DEVICE}")
print(f"  Total params   : {total_params:,}")
print(f"  Saved          : aftershocknet_init.pt | model_config.pkl")

# Step 5

In [ ]:
# STEP 5: Training Loop (Fixed — Normalised Loss + GPU)

import os, gc, math, pickle, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch_geometric.data import Batch
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

OUT_DIR  = "/kaggle/working/"
HORIZONS = ['1h', '6h', '12h', '24h', '72h']

# ── Load metadata ─────────────────────────────────────────────
with open(f"{OUT_DIR}dataset_meta.pkl", 'rb') as f:
    meta = pickle.load(f)
with open(f"{OUT_DIR}model_config.pkl", 'rb') as f:
    cfg = pickle.load(f)

chunk_index = meta['chunk_index']
N_common    = meta['N_common']
n_train     = meta['n_train']
n_val       = meta['n_val']
pos_weights = meta['pos_weights']

# ── Device ────────────────────────────────────────────────────
def cuda_works():
    if not torch.cuda.is_available():
        return False
    try:
        a = torch.ones(4, 4).cuda()
        _ = a @ a
        del a, _
        torch.cuda.empty_cache()
        return True
    except Exception:
        return False

DEVICE = torch.device('cuda' if cuda_works() else 'cpu')
print(f"Device   : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()

print(f"N_common : {N_common:,}  train : {n_train:,}  val : {n_val:,}")

# ── Label normalisation ───────────────────────────────────────
if 'label_norm' not in meta:
    print("Computing label normalisation stats...")
    sample = {h: {'occ': [], 'mag': [], 'lat': [], 'lon': []}
              for h in HORIZONS}
    for entry in chunk_index['w24'][:3]:
        part = torch.load(entry['path'], weights_only=False)
        for g in part:
            for h in HORIZONS:
                sample[h]['occ'].append(getattr(g, f'{h}_occ').item())
                sample[h]['mag'].append(getattr(g, f'{h}_mag').item())
                sample[h]['lat'].append(getattr(g, f'{h}_lat').item())
                sample[h]['lon'].append(getattr(g, f'{h}_lon').item())
        del part
        gc.collect()

    label_norm = {}
    print(f"\n{'Horizon':<8} {'LatStd':>8} {'LonStd':>8} "
          f"{'MagStd':>8} {'PosRate':>8}")
    print("-" * 44)
    for h in HORIZONS:
        occ  = torch.tensor(sample[h]['occ'])
        lat  = torch.tensor(sample[h]['lat'])
        lon  = torch.tensor(sample[h]['lon'])
        mag  = torch.tensor(sample[h]['mag'])
        mask = occ > 0
        lat_std = max(float(lat[mask].std()) if mask.sum() > 1 else 1.0, 1e-3)
        lon_std = max(float(lon[mask].std()) if mask.sum() > 1 else 1.0, 1e-3)
        mag_std = max(float(mag[mask].std()) if mask.sum() > 1 else 1.0, 1e-3)
        label_norm[h] = {
            'lat_std': lat_std, 'lon_std': lon_std, 'mag_std': mag_std}
        print(f"{h:<8} {lat_std:>8.3f} {lon_std:>8.3f} "
              f"{mag_std:>8.3f} {float(mask.float().mean()):>8.3f}")
    meta['label_norm'] = label_norm
    with open(f"{OUT_DIR}dataset_meta.pkl", 'wb') as f:
        pickle.dump(meta, f)
    print("  Label norm saved.\n")
else:
    label_norm = meta['label_norm']
    print("Label norm loaded from metadata.")
    for h in HORIZONS:
        print(f"  {h}: lat_std={label_norm[h]['lat_std']:.3f}  "
              f"lon_std={label_norm[h]['lon_std']:.3f}  "
              f"mag_std={label_norm[h]['mag_std']:.3f}")

# ── Index maps ────────────────────────────────────────────────
def build_index_map(w_name, chunk_index, n_common):
    index_map = []
    for entry in chunk_index[w_name]:
        part_len = entry['end'] - entry['start']
        for li in range(part_len):
            index_map.append((entry['path'], entry['start'] + li))
            if len(index_map) >= n_common:
                return index_map
    return index_map

print("\nBuilding index maps...")
idx_map_w24 = build_index_map('w24', chunk_index, N_common)
idx_map_w48 = build_index_map('w48', chunk_index, N_common)
idx_map_w72 = build_index_map('w72', chunk_index, N_common)
print(f"  w24:{len(idx_map_w24):,}  "
      f"w48:{len(idx_map_w48):,}  "
      f"w72:{len(idx_map_w72):,}")

# ── Dataset ───────────────────────────────────────────────────
class ChunkStreamDataset(Dataset):
    def __init__(self, map24, map48, map72, horizons, indices):
        self.map24    = map24
        self.map48    = map48
        self.map72    = map72
        self.horizons = horizons
        self.indices  = indices
        self._c24     = {}
        self._c48     = {}
        self._c72     = {}

    def _get(self, cache, path, local_idx):
        if path not in cache:
            cache.clear()
            gc.collect()
            cache[path] = torch.load(path, weights_only=False)
        return cache[path][local_idx]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        gi       = self.indices[i]
        p24, l24 = self.map24[gi]
        p48, l48 = self.map48[gi]
        p72, l72 = self.map72[gi]
        g24 = self._get(self._c24, p24, l24)
        g48 = self._get(self._c48, p48, l48)
        g72 = self._get(self._c72, p72, l72)
        parts = []
        for h in self.horizons:
            parts += [
                getattr(g24, f'{h}_occ'),
                getattr(g24, f'{h}_mag'),
                getattr(g24, f'{h}_lat'),
                getattr(g24, f'{h}_lon'),
            ]
        return g24, g48, g72, torch.cat(parts, dim=0)

def collate_fn(batch):
    g24, g48, g72, lbl = zip(*batch)
    return (Batch.from_data_list(list(g24)),
            Batch.from_data_list(list(g48)),
            Batch.from_data_list(list(g72)),
            torch.stack(lbl, dim=0))

train_ds = ChunkStreamDataset(
    idx_map_w24, idx_map_w48, idx_map_w72,
    HORIZONS, list(range(0, n_train)))
val_ds = ChunkStreamDataset(
    idx_map_w24, idx_map_w48, idx_map_w72,
    HORIZONS, list(range(n_train, n_train + n_val)))

BATCH_SIZE   = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn,
                          num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn,
                          num_workers=0)
print(f"  Train batches : {len(train_loader):,}")
print(f"  Val batches   : {len(val_loader):,}")

# ── Model ─────────────────────────────────────────────────────
class ETASPriorEmbedding(nn.Module):
    def __init__(self, n_etas=5, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_etas, 64), nn.LayerNorm(64),
            nn.GELU(), nn.Linear(64, embed_dim), nn.GELU())
        self.idx = [6, 7, 8, 9, 10]
    def forward(self, x):
        return self.net(x[:, self.idx])

class STGATBlock(nn.Module):
    def __init__(self, in_dim, out_dim, n_heads=4, edge_dim=4, dropout=0.1):
        super().__init__()
        self.gat  = GATv2Conv(in_dim, out_dim // n_heads, heads=n_heads,
                              edge_dim=edge_dim, concat=True, dropout=dropout)
        self.norm = nn.LayerNorm(out_dim)
        self.drop = nn.Dropout(dropout)
        self.res  = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
    def forward(self, x, ei, ea):
        return self.norm(self.drop(self.gat(x, ei, ea)) + self.res(x))

class MultiWindowGNNEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, n_layers=3,
                 n_heads=4, edge_dim=4, etas_embed_dim=32, dropout=0.1):
        super().__init__()
        self.etas      = ETASPriorEmbedding(5, etas_embed_dim)
        self.in_proj   = nn.Linear(in_dim + etas_embed_dim, hidden_dim)
        self.layers    = nn.ModuleList([
            STGATBlock(hidden_dim, hidden_dim, n_heads, edge_dim, dropout)
            for _ in range(n_layers)])
        self.pool_proj = nn.Linear(hidden_dim * 2, out_dim)
        self.fuse      = nn.Sequential(
            nn.Linear(out_dim * 3, out_dim * 2),
            nn.LayerNorm(out_dim * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(out_dim * 2, out_dim))
    def encode(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.in_proj(torch.cat([x, self.etas(x)], dim=1))
        for l in self.layers:
            x = l(x, ei, ea)
        return self.pool_proj(
            torch.cat([global_mean_pool(x, b), global_max_pool(x, b)], dim=1))
    def forward(self, g24, g48, g72):
        return self.fuse(
            torch.cat([self.encode(g24), self.encode(g48), self.encode(g72)], dim=1))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])

class TemporalTransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads=8, n_layers=4,
                 ffn_dim=512, dropout=0.1, n_horizons=5):
        super().__init__()
        self.pos  = PositionalEncoding(d_model, dropout=dropout)
        self.htok = nn.Parameter(torch.randn(n_horizons, d_model) * 0.02)
        self.tf   = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model, n_heads, ffn_dim, dropout,
                activation='gelu', batch_first=True, norm_first=True),
            num_layers=n_layers, norm=nn.LayerNorm(d_model))
    def forward(self, x):
        B   = x.size(0)
        seq = torch.cat([x.unsqueeze(1),
                         self.htok.unsqueeze(0).expand(B, -1, -1)], dim=1)
        return self.tf(self.pos(seq))[:, 1:, :]

class HorizonHead(nn.Module):
    def __init__(self, d_model, hidden=128, dropout=0.1):
        super().__init__()
        def mlp(o):
            return nn.Sequential(
                nn.Linear(d_model, hidden), nn.LayerNorm(hidden),
                nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, o))
        self.occ = mlp(1); self.mag = mlp(1); self.loc = mlp(2)
    def forward(self, x):
        return self.occ(x), self.mag(x), self.loc(x)

class AfterShockNet(nn.Module):
    def __init__(self, n_node_feat=16, n_edge_feat=4, gnn_hidden=128,
                 gnn_out=256, gnn_layers=3, gnn_heads=4, etas_embed=32,
                 tf_d_model=256, tf_heads=8, tf_layers=4, tf_ffn=512,
                 head_hidden=128, dropout=0.1, n_horizons=5):
        super().__init__()
        self.gnn  = MultiWindowGNNEncoder(
            n_node_feat, gnn_hidden, gnn_out,
            gnn_layers, gnn_heads, n_edge_feat, etas_embed, dropout)
        self.proj = (nn.Linear(gnn_out, tf_d_model)
                     if gnn_out != tf_d_model else nn.Identity())
        self.tf   = TemporalTransformerEncoder(
            tf_d_model, tf_heads, tf_layers, tf_ffn, dropout, n_horizons)
        self.heads = nn.ModuleList([
            HorizonHead(tf_d_model, head_hidden, dropout)
            for _ in range(n_horizons)])
        self.horizons = HORIZONS
    def forward(self, g24, g48, g72):
        h   = self.tf(self.proj(self.gnn(g24, g48, g72)))
        out = {}
        for i, hn in enumerate(self.horizons):
            o, m, l = self.heads[i](h[:, i, :])
            out[hn] = {'occ_logit': o, 'mag': m, 'loc': l}
        return out

model = AfterShockNet(**{k: cfg[k] for k in [
    'n_node_feat', 'n_edge_feat', 'gnn_hidden', 'gnn_out', 'gnn_layers',
    'gnn_heads', 'etas_embed', 'tf_d_model', 'tf_heads', 'tf_layers',
    'tf_ffn', 'head_hidden', 'dropout', 'n_horizons']}).to(DEVICE)

print(f"Model params : {sum(p.numel() for p in model.parameters()):,}")

# ── Loss ──────────────────────────────────────────────────────
def compute_loss(preds, labels, pos_weights, horizons, device, label_norm):
    total    = torch.tensor(0.0, device=device)
    loss_log = {}
    for i, h in enumerate(horizons):
        base    = i * 4
        occ_lbl = labels[:, base].unsqueeze(1)
        mag_lbl = labels[:, base + 1].unsqueeze(1)
        lat_lbl = labels[:, base + 2].unsqueeze(1)
        lon_lbl = labels[:, base + 3].unsqueeze(1)
        pw      = pos_weights[h].to(device)
        l_occ   = F.binary_cross_entropy_with_logits(
            preds[h]['occ_logit'], occ_lbl, pos_weight=pw)
        mask = occ_lbl.squeeze(1).bool()
        if mask.sum() > 0:
            mag_s = label_norm[h]['mag_std']
            lat_s = label_norm[h]['lat_std']
            lon_s = label_norm[h]['lon_std']
            l_mag = F.mse_loss(
                preds[h]['mag'][mask] / mag_s,
                mag_lbl[mask] / mag_s)
            lat_pred = preds[h]['loc'][mask][:, 0:1] / lat_s
            lon_pred = preds[h]['loc'][mask][:, 1:2] / lon_s
            lat_true = lat_lbl[mask] / lat_s
            lon_true = lon_lbl[mask] / lon_s
            l_loc = F.mse_loss(
                torch.cat([lat_pred, lon_pred], dim=1),
                torch.cat([lat_true, lon_true], dim=1))
        else:
            l_mag = torch.tensor(0.0, device=device)
            l_loc = torch.tensor(0.0, device=device)
        h_loss = l_occ + 0.3 * l_mag + 0.3 * l_loc
        total  = total + h_loss
        loss_log[h] = {
            'occ': l_occ.item(),
            'mag': l_mag.item(),
            'loc': l_loc.item()}
    return total, loss_log

# ── Metrics (pure torch — no numpy) ──────────────────────────
def compute_metrics(preds, labels, horizons):
    """
    All operations in pure PyTorch — no numpy dependency.
    Returns dict of metrics per horizon.
    """
    metrics = {}
    for i, h in enumerate(horizons):
        base     = i * 4
        occ_lbl  = labels[:, base].float().cpu()
        mag_lbl  = labels[:, base + 1].float().cpu()
        lat_lbl  = labels[:, base + 2].float().cpu()
        lon_lbl  = labels[:, base + 3].float().cpu()

        occ_prob = torch.sigmoid(
            preds[h]['occ_logit'].float()).squeeze(1).detach().cpu()
        occ_pred = (occ_prob >= 0.5).float()
        mag_pred = preds[h]['mag'].float().squeeze(1).detach().cpu()
        loc_pred = preds[h]['loc'].float().detach().cpu()

        tp = ((occ_pred == 1) & (occ_lbl == 1)).sum().float()
        fp = ((occ_pred == 1) & (occ_lbl == 0)).sum().float()
        fn = ((occ_pred == 0) & (occ_lbl == 1)).sum().float()
        prec   = tp / (tp + fp).clamp(min=1)
        recall = tp / (tp + fn).clamp(min=1)
        f1     = (2 * prec * recall / (prec + recall + 1e-8))

        mask     = occ_lbl == 1
        n_pos    = mask.sum().item()

        if n_pos > 0:
            mag_rmse = float(((mag_pred[mask] - mag_lbl[mask])**2
                               ).mean().sqrt())
            loc_err  = ((loc_pred[mask, 0] - lat_lbl[mask])**2 +
                        (loc_pred[mask, 1] - lon_lbl[mask])**2).sqrt()
            loc_mae  = float(loc_err.mean())
        else:
            mag_rmse = 0.0
            loc_mae  = 0.0

        metrics[h] = {
            'f1':        float(f1),
            'precision': float(prec),
            'recall':    float(recall),
            'mag_rmse':  mag_rmse,
            'loc_mae':   loc_mae,
        }
    return metrics

# ── Optimiser & Scheduler ─────────────────────────────────────
EPOCHS = 30
LR_MAX = 1e-4
WD     = 1e-4

optimizer = AdamW(model.parameters(), lr=LR_MAX, weight_decay=WD)
scheduler = OneCycleLR(
    optimizer, max_lr=LR_MAX,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.15,
    anneal_strategy='cos')

# ── Training Loop ─────────────────────────────────────────────
history        = {'train_loss': [], 'val_loss': [], 'val_metrics': []}
best_val_loss  = float('inf')
patience       = 15
patience_count = 0

print(f"\nTraining — {EPOCHS} epochs on {DEVICE}")
print(f"  LR={LR_MAX}  WD={WD}  patience={patience}")
print(f"  Loss: normalised BCE + 0.3*MSE_mag + 0.3*MSE_loc")
print(f"\n{'Ep':>4} {'Train':>8} {'Val':>8} "
      f"{'1h-F1':>7} {'6h-F1':>7} {'12h-F1':>7} "
      f"{'24h-F1':>7} {'72h-F1':>7}  Time")
print("-" * 72)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train ─────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for bg24, bg48, bg72, lbl in train_loader:
        bg24 = bg24.to(DEVICE)
        bg48 = bg48.to(DEVICE)
        bg72 = bg72.to(DEVICE)
        lbl  = lbl.to(DEVICE)
        optimizer.zero_grad()
        preds   = model(bg24, bg48, bg72)
        loss, _ = compute_loss(
            preds, lbl, pos_weights, HORIZONS, DEVICE, label_norm)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        del bg24, bg48, bg72, lbl, preds, loss
    train_loss /= len(train_loader)

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

    # ── Validate ──────────────────────────────────────────────
    model.eval()
    val_loss   = 0.0
    all_preds  = {h: {'occ_logit': [], 'mag': [], 'loc': []}
                  for h in HORIZONS}
    all_labels = []

    with torch.no_grad():
        for bg24, bg48, bg72, lbl in val_loader:
            bg24 = bg24.to(DEVICE)
            bg48 = bg48.to(DEVICE)
            bg72 = bg72.to(DEVICE)
            lbl  = lbl.to(DEVICE)
            preds    = model(bg24, bg48, bg72)
            loss, _  = compute_loss(
                preds, lbl, pos_weights, HORIZONS, DEVICE, label_norm)
            val_loss += loss.item()
            all_labels.append(lbl.float().cpu())
            for h in HORIZONS:
                all_preds[h]['occ_logit'].append(
                    preds[h]['occ_logit'].float().cpu())
                all_preds[h]['mag'].append(
                    preds[h]['mag'].float().cpu())
                all_preds[h]['loc'].append(
                    preds[h]['loc'].float().cpu())
            del bg24, bg48, bg72, lbl, preds, loss

    val_loss  /= len(val_loader)
    all_labels = torch.cat(all_labels, dim=0)
    all_preds  = {h: {k: torch.cat(v, dim=0)
                      for k, v in all_preds[h].items()}
                  for h in HORIZONS}

    metrics = compute_metrics(all_preds, all_labels, HORIZONS)
    elapsed = time.time() - t0

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_metrics'].append(metrics)

    print(f"{epoch:>4}  {train_loss:>8.4f}  {val_loss:>8.4f}  "
          f"{metrics['1h']['f1']:>7.4f}  "
          f"{metrics['6h']['f1']:>7.4f}  "
          f"{metrics['12h']['f1']:>7.4f}  "
          f"{metrics['24h']['f1']:>7.4f}  "
          f"{metrics['72h']['f1']:>7.4f}  "
          f"{elapsed:>5.1f}s")

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        patience_count = 0
        torch.save({
            'epoch':           epoch,
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss':        val_loss,
            'metrics':         metrics},
            f"{OUT_DIR}best_model.pt")
        print(f"       ✓ best saved (val={val_loss:.4f})")
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

# ── Save ──────────────────────────────────────────────────────
torch.save(model.state_dict(), f"{OUT_DIR}final_model.pt")
with open(f"{OUT_DIR}train_history.pkl", 'wb') as f:
    pickle.dump(history, f)

meta['device'] = str(DEVICE)
with open(f"{OUT_DIR}dataset_meta.pkl", 'wb') as f:
    pickle.dump(meta, f)

print(f"\nStep 5 complete.")
print(f"  Best val loss : {best_val_loss:.4f}")
print(f"  Device        : {DEVICE}")
print(f"  Saved         : best_model.pt | final_model.pt | train_history.pkl")

# step 5.1

In [ ]:
# STEP 5.1: Baseline Models Training (Fixed)
# XGBoost | LSTM | Transformer | ETAS


import os, gc, pickle, time, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

OUT_DIR      = "/kaggle/working/"
BASELINE_DIR = f"{OUT_DIR}baselines/"
os.makedirs(BASELINE_DIR, exist_ok=True)
HORIZONS = ['1h', '6h', '12h', '24h', '72h']

def cuda_works():
    if not torch.cuda.is_available(): return False
    try:
        a=torch.ones(4,4).cuda(); _=a@a
        del a,_; torch.cuda.empty_cache(); return True
    except: return False

DEVICE = torch.device('cuda' if cuda_works() else 'cpu')
print(f"Device : {DEVICE}")

# ── Load metadata & features ──────────────────────────────────
with open(f"{OUT_DIR}dataset_meta.pkl",'rb') as f:
    meta = pickle.load(f)
with open(f"{OUT_DIR}labels.pkl",'rb') as f:
    labels = pickle.load(f)

eq         = pd.read_csv(f"{OUT_DIR}eq_features.csv")
eq_arr_raw = __import__('numpy').load(f"{OUT_DIR}eq_arr_raw.npy")
import numpy as np

N_common = meta['N_common']
n_train  = meta['n_train']
n_val    = meta['n_val']
n_test   = meta['n_test']
pos_weights = meta['pos_weights']
label_norm  = meta['label_norm']

FEATURE_COLS = [
    'latitude','longitude','depth','mag',
    'dist_fault_km','dist_pb_km',
    'etas_lambda','etas_mu','etas_n_prior','etas_max_mag','etas_energy',
    'hour_sin','hour_cos','doy_sin','doy_cos','dt_prev_s'
]

# ── Build flat arrays ─────────────────────────────────────────
print("Building flat feature/label arrays...")
STEP = meta.get('step', 20)
N_eq = len(eq_arr_raw)
anchor_indices = list(range(5, N_eq, STEP))[:N_common]

X_all = eq[FEATURE_COLS].values.astype(np.float32)

Y = np.zeros((N_common, 20), dtype=np.float32)
for i, h in enumerate(HORIZONS):
    base = i * 4
    for j, anc in enumerate(anchor_indices):
        Y[j, base]   = labels[h]['occurrence'][anc]
        Y[j, base+1] = labels[h]['magnitude'][anc]
        Y[j, base+2] = labels[h]['lat'][anc]
        Y[j, base+3] = labels[h]['lon'][anc]

X = X_all[anchor_indices].astype(np.float32)
X_train = X[:n_train].copy()
X_val   = X[n_train:n_train+n_val].copy()
X_test  = X[n_train+n_val:].copy()
Y_train = Y[:n_train].copy()
Y_val   = Y[n_train:n_train+n_val].copy()
Y_test  = Y[n_train+n_val:].copy()

print(f"  X_train:{X_train.shape}  X_test:{X_test.shape}")

# ── Metric helpers (fixed: use torch.from_numpy) ─────────────
def to_tensor(x):
    if isinstance(x, torch.Tensor):
        return x.float()
    return torch.tensor(x, dtype=torch.float32)

def classification_metrics(y_true, y_prob, threshold=0.5):
    yt   = to_tensor(y_true)
    yp   = to_tensor(y_prob)
    pred = (yp >= threshold).float()
    tp   = ((pred==1)&(yt==1)).sum().float()
    fp   = ((pred==1)&(yt==0)).sum().float()
    fn   = ((pred==0)&(yt==1)).sum().float()
    tn   = ((pred==0)&(yt==0)).sum().float()
    prec = float(tp/(tp+fp).clamp(min=1))
    rec  = float(tp/(tp+fn).clamp(min=1))
    f1   = float(2*prec*rec/max(prec+rec,1e-8))
    acc  = float((tp+tn)/len(yt))
    n_pos= int((yt==1).sum()); n_neg=int((yt==0).sum())
    if n_pos==0 or n_neg==0:
        auc=0.5; prauc=float(prec)
    else:
        sidx  = torch.argsort(yp)
        rank  = torch.arange(1,len(yt)+1,dtype=torch.float32)
        rs    = rank[torch.argsort(yp)][yt[torch.argsort(yp)]==1].sum()
        auc   = float(((rs-n_pos*(n_pos+1)/2)/(n_pos*n_neg)).clamp(0,1))
        sidx2 = torch.argsort(yp,descending=True)
        ys    = yt[sidx2]; tc=torch.cumsum(ys,0)
        pk    = tc/torch.arange(1,len(yt)+1,dtype=torch.float32)
        rk    = tc/max(n_pos,1)
        rd    = torch.diff(rk,prepend=torch.zeros(1))
        prauc = float((pk*rd).sum().clamp(0,1))
    return {'acc':acc,'prec':prec,'recall':rec,'f1':f1,
            'auc':auc,'prauc':prauc}

def regression_metrics(y_true, y_pred, mask=None):
    yt = to_tensor(y_true)
    yp = to_tensor(y_pred)
    if mask is not None:
        m = to_tensor(mask).bool()
        if m.sum()==0: return {'rmse':0.0,'mae':0.0}
        yt=yt[m]; yp=yp[m]
    rmse = float(((yp-yt)**2).mean().sqrt())
    mae  = float((yp-yt).abs().mean())
    return {'rmse':rmse,'mae':mae}

def compute_flat_loss(preds, labels, pos_weights, horizons,
                      device, label_norm):
    total = torch.tensor(0.0, device=device)
    for i,h in enumerate(horizons):
        base    = i*4
        occ_lbl = labels[:,base].unsqueeze(1)
        mag_lbl = labels[:,base+1].unsqueeze(1)
        lat_lbl = labels[:,base+2].unsqueeze(1)
        lon_lbl = labels[:,base+3].unsqueeze(1)
        pw      = pos_weights[h].to(device)
        l_occ   = F.binary_cross_entropy_with_logits(
            preds[h]['occ_logit'], occ_lbl, pos_weight=pw)
        mask = occ_lbl.squeeze(1).bool()
        if mask.sum()>0:
            ms=label_norm[h]['mag_std']
            ls=label_norm[h]['lat_std']
            lo=label_norm[h]['lon_std']
            l_mag=F.mse_loss(preds[h]['mag'][mask]/ms, mag_lbl[mask]/ms)
            l_loc=F.mse_loss(
                torch.cat([preds[h]['loc'][mask][:,0:1]/ls,
                           preds[h]['loc'][mask][:,1:2]/lo],dim=1),
                torch.cat([lat_lbl[mask]/ls,
                           lon_lbl[mask]/lo],dim=1))
        else:
            l_mag=torch.tensor(0.0,device=device)
            l_loc=torch.tensor(0.0,device=device)
        total = total + l_occ + 0.3*l_mag + 0.3*l_loc
    return total


# MODEL 1: XGBoost
print("\n"+"="*55)
print("Training XGBoost baselines...")
print("="*55)

try:
    import xgboost as xgb
except ImportError:
    import subprocess
    subprocess.run(['pip','install','xgboost','-q'])
    import xgboost as xgb

xgb_results={}
xgb_preds_all={}

for i,h in enumerate(HORIZONS):
    base = i*4
    y_tr_occ = Y_train[:,base].astype(np.float32)
    y_va_occ = Y_val[:,base].astype(np.float32)
    y_te_occ = Y_test[:,base].astype(np.float32)
    y_tr_mag = Y_train[:,base+1].astype(np.float32)
    y_te_mag = Y_test[:,base+1].astype(np.float32)

    t0 = time.time()

    clf = xgb.XGBClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss', tree_method='hist',
        device='cuda' if DEVICE.type=='cuda' else 'cpu',
        random_state=42, verbosity=0)
    clf.fit(X_train, y_tr_occ,
            eval_set=[(X_val, y_va_occ)],
            verbose=False)

    pos_mask_tr = y_tr_occ == 1
    pos_mask_te = y_te_occ == 1

    reg = xgb.XGBRegressor(
        n_estimators=300, max_depth=6,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, tree_method='hist',
        device='cuda' if DEVICE.type=='cuda' else 'cpu',
        random_state=42, verbosity=0)
    if pos_mask_tr.sum() > 10:
        reg.fit(X_train[pos_mask_tr], y_tr_mag[pos_mask_tr])
        mag_pred = reg.predict(X_test).astype(np.float32)
    else:
        mag_pred = np.zeros(len(X_test), dtype=np.float32)

    occ_prob = clf.predict_proba(X_test)[:,1].astype(np.float32)

    cm = classification_metrics(y_te_occ, occ_prob)
    rm = regression_metrics(y_te_mag, mag_pred, mask=pos_mask_te)

    elapsed = time.time()-t0
    xgb_results[h]   = {**cm,**rm}
    xgb_preds_all[h] = {'occ_prob': occ_prob, 'mag_pred': mag_pred}
    print(f"  XGB {h:>4s}  F1={cm['f1']:.4f}  AUC={cm['auc']:.4f}  "
          f"MagRMSE={rm['rmse']:.4f}  ({elapsed:.1f}s)")

with open(f"{BASELINE_DIR}xgb_results.pkl",'wb') as f:
    pickle.dump({'results':xgb_results,'preds':xgb_preds_all,
                 'Y_test':Y_test},f)
print("  Saved xgb_results.pkl")
gc.collect()


# MODEL 2: LSTM
print("\n"+"="*55)
print("Training LSTM baseline...")
print("="*55)

SEQ_LEN    = 20
LSTM_BATCH = 256
LSTM_EPOCH = 30

def build_sequences(X_feat, anchors, seq_len):
    seqs = []
    for anc in anchors:
        start = max(0, anc-seq_len)
        seq   = X_feat[start:anc+1]
        if len(seq) < seq_len:
            pad = np.zeros((seq_len-len(seq), X_feat.shape[1]),
                           dtype=np.float32)
            seq = np.vstack([pad,seq])
        else:
            seq = seq[-seq_len:]
        seqs.append(seq)
    return np.stack(seqs,axis=0).astype(np.float32)

print("  Building sequences...")
X_seq_tr = build_sequences(X_all, anchor_indices[:n_train], SEQ_LEN)
X_seq_va = build_sequences(X_all, anchor_indices[n_train:n_train+n_val], SEQ_LEN)
X_seq_te = build_sequences(X_all, anchor_indices[n_train+n_val:], SEQ_LEN)
print(f"  X_seq_tr:{X_seq_tr.shape}")

class LSTMForecaster(nn.Module):
    def __init__(self, input_dim=16, hidden=128,
                 n_layers=2, dropout=0.2, n_horizons=5):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, n_layers,
                            batch_first=True, dropout=dropout)
        self.norm = nn.LayerNorm(hidden)
        self.heads = nn.ModuleList([
            nn.ModuleDict({
                'occ': nn.Sequential(nn.Linear(hidden,64),
                                     nn.ReLU(),nn.Linear(64,1)),
                'mag': nn.Sequential(nn.Linear(hidden,64),
                                     nn.ReLU(),nn.Linear(64,1)),
                'loc': nn.Sequential(nn.Linear(hidden,64),
                                     nn.ReLU(),nn.Linear(64,2)),
            }) for _ in range(n_horizons)])

    def forward(self, x):
        _,(h,_) = self.lstm(x)
        feat    = self.norm(h[-1])
        preds   = {}
        for i,hn in enumerate(HORIZONS):
            preds[hn]={'occ_logit':self.heads[i]['occ'](feat),
                       'mag':      self.heads[i]['mag'](feat),
                       'loc':      self.heads[i]['loc'](feat)}
        return preds

lstm_model = LSTMForecaster().to(DEVICE)

X_tr_t = torch.tensor(X_seq_tr, dtype=torch.float32)
Y_tr_t = torch.tensor(Y_train, dtype=torch.float32)
X_va_t = torch.tensor(X_seq_va, dtype=torch.float32)
Y_va_t = torch.tensor(Y_val, dtype=torch.float32)
X_te_t = torch.tensor(X_seq_te, dtype=torch.float32)

tr_ds = TensorDataset(X_tr_t, Y_tr_t)
va_ds = TensorDataset(X_va_t, Y_va_t)
tr_ldr= DataLoader(tr_ds, batch_size=LSTM_BATCH, shuffle=False)
va_ldr= DataLoader(va_ds, batch_size=LSTM_BATCH, shuffle=False)

opt_l = AdamW(lstm_model.parameters(), lr=1e-3, weight_decay=1e-4)
sch_l = OneCycleLR(opt_l, max_lr=1e-3,
                   steps_per_epoch=len(tr_ldr),
                   epochs=LSTM_EPOCH, pct_start=0.1)

best_lstm_val = float('inf')
print(f"  Training {LSTM_EPOCH} epochs...")

for epoch in range(1, LSTM_EPOCH+1):
    lstm_model.train()
    tr_loss = 0.0
    for xb,yb in tr_ldr:
        xb=xb.to(DEVICE); yb=yb.to(DEVICE)
        opt_l.zero_grad()
        preds=lstm_model(xb)
        loss =compute_flat_loss(preds,yb,pos_weights,
                                HORIZONS,DEVICE,label_norm)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(),1.0)
        opt_l.step(); sch_l.step()
        tr_loss+=loss.item()
        del xb,yb,preds,loss
    tr_loss/=len(tr_ldr)

    lstm_model.eval()
    va_loss=0.0
    with torch.no_grad():
        for xb,yb in va_ldr:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE)
            preds=lstm_model(xb)
            va_loss+=compute_flat_loss(preds,yb,pos_weights,
                                       HORIZONS,DEVICE,label_norm).item()
            del xb,yb,preds
    va_loss/=len(va_ldr)

    if va_loss<best_lstm_val:
        best_lstm_val=va_loss
        torch.save(lstm_model.state_dict(),
                   f"{BASELINE_DIR}lstm_best.pt")

    if epoch%5==0:
        print(f"    Ep {epoch:>3}  train={tr_loss:.4f}  val={va_loss:.4f}")
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    gc.collect()

# Inference
lstm_model.load_state_dict(
    torch.load(f"{BASELINE_DIR}lstm_best.pt",weights_only=False))
lstm_model.eval()
lstm_preds_all={h:{'occ_prob':[],'mag':[]} for h in HORIZONS}
te_ds = TensorDataset(
    X_te_t,
    torch.tensor(Y_test, dtype=torch.float32)
)
te_ldr = DataLoader(te_ds, batch_size=LSTM_BATCH, shuffle=False)

with torch.no_grad():
    for xb,_ in te_ldr:
        xb=xb.to(DEVICE)
        preds=lstm_model(xb)
        for h in HORIZONS:
            lstm_preds_all[h]['occ_prob'].append(
                torch.sigmoid(preds[h]['occ_logit']).squeeze(1).float().cpu())
            lstm_preds_all[h]['mag'].append(
                preds[h]['mag'].squeeze(1).float().cpu())
        del xb,preds
    if DEVICE.type=='cuda': torch.cuda.empty_cache()

lstm_results={}
for i,h in enumerate(HORIZONS):
    base     = i*4
    occ_prob = torch.cat(lstm_preds_all[h]['occ_prob']).float()
    mag_pred = torch.cat(lstm_preds_all[h]['mag']).float()
    occ_lbl  = Y_test[:,base]
    mag_lbl  = Y_test[:,base+1]
    pos_mask = occ_lbl==1
    cm = classification_metrics(occ_lbl, occ_prob)
    rm = regression_metrics(mag_lbl, mag_pred, mask=pos_mask)
    lstm_results[h]={**cm,**rm}
    lstm_preds_all[h] = {
    'occ_prob': occ_prob.cpu().tolist(),
    'mag_pred': mag_pred.cpu().tolist()
}
    print(f"  LSTM {h:>4s}  F1={cm['f1']:.4f}  AUC={cm['auc']:.4f}  "
          f"MagRMSE={rm['rmse']:.4f}")

with open(f"{BASELINE_DIR}lstm_results.pkl",'wb') as f:
    pickle.dump({'results':lstm_results,'preds':lstm_preds_all,
                 'Y_test':Y_test},f)
print("  Saved lstm_results.pkl")

del lstm_model,X_seq_tr,X_seq_va,X_seq_te
del X_tr_t,X_va_t,X_te_t,tr_ds,va_ds,te_ds
gc.collect()
if DEVICE.type=='cuda': torch.cuda.empty_cache()


# MODEL 3: Standalone Transformer
print("\n"+"="*55)
print("Training Standalone Transformer baseline...")
print("="*55)

class StandaloneTransformer(nn.Module):
    def __init__(self, input_dim=16, d_model=128, n_heads=4,
                 n_layers=3, ffn_dim=256, dropout=0.1, n_horizons=5):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pos  = nn.Parameter(torch.randn(1,1,d_model)*0.02)
        enc = nn.TransformerEncoderLayer(d_model,n_heads,ffn_dim,
              dropout,activation='gelu',batch_first=True,norm_first=True)
        self.tf   = nn.TransformerEncoder(enc,num_layers=n_layers,
                                          norm=nn.LayerNorm(d_model))
        self.heads= nn.ModuleList([
            nn.ModuleDict({
                'occ': nn.Sequential(nn.Linear(d_model,64),
                                     nn.GELU(),nn.Linear(64,1)),
                'mag': nn.Sequential(nn.Linear(d_model,64),
                                     nn.GELU(),nn.Linear(64,1)),
                'loc': nn.Sequential(nn.Linear(d_model,64),
                                     nn.GELU(),nn.Linear(64,2)),
            }) for _ in range(n_horizons)])

    def forward(self,x):
        x   = self.proj(x).unsqueeze(1)+self.pos
        out = self.tf(x).squeeze(1)
        preds={}
        for i,hn in enumerate(HORIZONS):
            preds[hn]={'occ_logit':self.heads[i]['occ'](out),
                       'mag':      self.heads[i]['mag'](out),
                       'loc':      self.heads[i]['loc'](out)}
        return preds

tf_model  = StandaloneTransformer().to(DEVICE)
TF_EPOCH  = 30
TF_BATCH  = 256

X_tr_f = torch.tensor(X_train, dtype=torch.float32)
Y_tr_f = torch.tensor(Y_train, dtype=torch.float32)
X_va_f = torch.tensor(X_val, dtype=torch.float32)
Y_va_f = torch.tensor(Y_val, dtype=torch.float32)
X_te_f = torch.tensor(X_test, dtype=torch.float32)

tr_ds_tf = TensorDataset(X_tr_f,Y_tr_f)
va_ds_tf = TensorDataset(X_va_f,Y_va_f)
tr_ldr_tf= DataLoader(tr_ds_tf,batch_size=TF_BATCH,shuffle=False)
va_ldr_tf= DataLoader(va_ds_tf,batch_size=TF_BATCH,shuffle=False)

opt_tf = AdamW(tf_model.parameters(),lr=1e-3,weight_decay=1e-4)
sch_tf = OneCycleLR(opt_tf,max_lr=1e-3,
                    steps_per_epoch=len(tr_ldr_tf),
                    epochs=TF_EPOCH,pct_start=0.1)

best_tf_val=float('inf')
print(f"  Training {TF_EPOCH} epochs...")

for epoch in range(1,TF_EPOCH+1):
    tf_model.train()
    tr_loss=0.0
    for xb,yb in tr_ldr_tf:
        xb=xb.to(DEVICE); yb=yb.to(DEVICE)
        opt_tf.zero_grad()
        preds=tf_model(xb)
        loss =compute_flat_loss(preds,yb,pos_weights,
                                HORIZONS,DEVICE,label_norm)
        loss.backward()
        nn.utils.clip_grad_norm_(tf_model.parameters(),1.0)
        opt_tf.step(); sch_tf.step()
        tr_loss+=loss.item()
        del xb,yb,preds,loss
    tr_loss/=len(tr_ldr_tf)

    tf_model.eval()
    va_loss=0.0
    with torch.no_grad():
        for xb,yb in va_ldr_tf:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE)
            preds=tf_model(xb)
            va_loss+=compute_flat_loss(preds,yb,pos_weights,
                                       HORIZONS,DEVICE,label_norm).item()
            del xb,yb,preds
    va_loss/=len(va_ldr_tf)

    if va_loss<best_tf_val:
        best_tf_val=va_loss
        torch.save(tf_model.state_dict(),
                   f"{BASELINE_DIR}transformer_best.pt")

    if epoch%5==0:
        print(f"    Ep {epoch:>3}  train={tr_loss:.4f}  val={va_loss:.4f}")
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    gc.collect()

# Inference
tf_model.load_state_dict(
    torch.load(f"{BASELINE_DIR}transformer_best.pt",weights_only=False))
tf_model.eval()
tf_preds_all={h:{'occ_prob':[],'mag':[]} for h in HORIZONS}
te_ds_tf = TensorDataset(
    X_te_f,
    torch.tensor(Y_test, dtype=torch.float32)
)
te_ldr_tf= DataLoader(te_ds_tf,batch_size=TF_BATCH,shuffle=False)

with torch.no_grad():
    for xb,_ in te_ldr_tf:
        xb=xb.to(DEVICE)
        preds=tf_model(xb)
        for h in HORIZONS:
            tf_preds_all[h]['occ_prob'].append(
                torch.sigmoid(preds[h]['occ_logit']).squeeze(1).float().cpu())
            tf_preds_all[h]['mag'].append(
                preds[h]['mag'].squeeze(1).float().cpu())
        del xb,preds
    if DEVICE.type=='cuda': torch.cuda.empty_cache()

tf_results={}
for i,h in enumerate(HORIZONS):
    base     = i*4
    occ_prob = torch.cat(tf_preds_all[h]['occ_prob']).float()
    mag_pred = torch.cat(tf_preds_all[h]['mag']).float()
    occ_lbl  = Y_test[:,base]
    mag_lbl  = Y_test[:,base+1]
    pos_mask = occ_lbl==1
    cm = classification_metrics(occ_lbl, occ_prob)
    rm = regression_metrics(mag_lbl, mag_pred, mask=pos_mask)
    tf_results[h]={**cm,**rm}
    tf_preds_all[h]={'occ_prob':occ_prob,'mag_pred':mag_pred}
    print(f"  TF   {h:>4s}  F1={cm['f1']:.4f}  AUC={cm['auc']:.4f}  "
          f"MagRMSE={rm['rmse']:.4f}")

with open(f"{BASELINE_DIR}transformer_results.pkl",'wb') as f:
    pickle.dump({'results':tf_results,'preds':tf_preds_all,
                 'Y_test':Y_test},f)
print("  Saved transformer_results.pkl")

del tf_model,tr_ds_tf,va_ds_tf,te_ds_tf
gc.collect()
if DEVICE.type=='cuda': torch.cuda.empty_cache()


# MODEL 4: ETAS Baseline
print("\n"+"="*55)
print("Computing ETAS baseline...")
print("="*55)

etas_col    = FEATURE_COLS.index('etas_lambda')
mag_col_idx = FEATURE_COLS.index('etas_max_mag')

etas_lambda_t = torch.tensor(
    X_test[:, etas_col],
    dtype=torch.float32
)
etas_prob_raw = torch.sigmoid(etas_lambda_t)

etas_results  ={}
etas_preds_all={}

for i,h in enumerate(HORIZONS):
    base     = i*4
    occ_lbl  = Y_test[:,base].astype(np.float32)
    mag_lbl  = Y_test[:,base+1].astype(np.float32)
    pos_rate = float(occ_lbl.mean())
    scale    = pos_rate/max(float(etas_prob_raw.mean()),1e-6)
    occ_prob = (etas_prob_raw * scale).clamp(0, 1).float()
    mag_pred = X_test[:,mag_col_idx].astype(np.float32)
    pos_mask = occ_lbl==1
    cm = classification_metrics(occ_lbl, occ_prob)
    rm = regression_metrics(mag_lbl, mag_pred, mask=pos_mask)
    etas_results[h]   ={**cm,**rm}
    etas_preds_all[h] ={'occ_prob':occ_prob,'mag_pred':mag_pred}
    print(f"  ETAS {h:>4s}  F1={cm['f1']:.4f}  AUC={cm['auc']:.4f}  "
          f"MagRMSE={rm['rmse']:.4f}")

with open(f"{BASELINE_DIR}etas_results.pkl",'wb') as f:
    pickle.dump({'results':etas_results,'preds':etas_preds_all,
                 'Y_test':Y_test},f)
print("  Saved etas_results.pkl")


# SUMMARY
print("\n"+"="*65)
print("BASELINE COMPARISON SUMMARY — F1 Score")
print("="*65)
print(f"{'Model':<20} "+"  ".join(f"{h:>7}" for h in HORIZONS))
print("-"*55)

all_model_results={
    'ETAS':        etas_results,
    'XGBoost':     xgb_results,
    'LSTM':        lstm_results,
    'Transformer': tf_results,
}
for mn,res in all_model_results.items():
    f1s=[f"{res[h]['f1']:.4f}" for h in HORIZONS]
    print(f"  {mn:<18} "+"  ".join(f"{f:>7}" for f in f1s))
print("-"*55)
print("  AfterShockNet        — see best_model.pt")

# Save summary CSV
rows=[]
for mn,res in all_model_results.items():
    for h in HORIZONS:
        r=res[h]
        rows.append({
            'Model':mn,'Horizon':h,
            'F1':      round(r['f1'],4),
            'AUC':     round(r['auc'],4),
            'PR-AUC':  round(r.get('prauc',0),4),
            'Prec':    round(r['prec'],4),
            'Recall':  round(r['recall'],4),
            'Acc':     round(r['acc'],4),
            'MagRMSE': round(r['rmse'],4),
            'MagMAE':  round(r['mae'],4),
        })

summary_df=pd.DataFrame(rows)
summary_df.to_csv(f"{BASELINE_DIR}baseline_summary.csv",index=False)
print(f"\n{summary_df.to_string(index=False)}")

# Save combined for Step 6
baseline_meta={
    'etas_results':  etas_results,
    'xgb_results':   xgb_results,
    'lstm_results':  lstm_results,
    'tf_results':    tf_results,
    'etas_preds':    etas_preds_all,
    'xgb_preds':     xgb_preds_all,
    'lstm_preds':    lstm_preds_all,
    'tf_preds':      tf_preds_all,
    'Y_test':        Y_test,
    'anchor_indices_test': anchor_indices[n_train+n_val:],
}
with open(f"{BASELINE_DIR}all_baselines.pkl",'wb') as f:
    pickle.dump(baseline_meta,f)

print(f"\nStep 5.1 complete.")
print(f"  Models : ETAS | XGBoost | LSTM | Transformer")
print(f"  Saved  : {BASELINE_DIR}")
print(f"\nReady for Step 6.")

# Step 6

In [ ]:
# STEP 6: Evaluation, Baseline Comparison, Visualization,
#         Statistical Significance, Ablation Study,
#         Tectonic Zone Stratification, M>=6 Sequences


import os, gc, math, pickle, struct, re, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Batch, Data
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
random.seed(42)

OUT_DIR      = "/kaggle/working/"
BASELINE_DIR = f"{OUT_DIR}baselines/"
FIG_DIR      = f"{OUT_DIR}figures/"
os.makedirs(FIG_DIR, exist_ok=True)

HORIZONS   = ['1h','6h','12h','24h','72h']
MODEL_NAMES= ['ETAS','XGBoost','LSTM','Transformer','AfterShockNet']
COLORS_MODEL = {
    'ETAS':         '#90A4AE',
    'XGBoost':      '#FF8F00',
    'LSTM':         '#43A047',
    'Transformer':  '#1E88E5',
    'AfterShockNet':'#E53935',
}
H_COLORS = plt.cm.tab10([0,1,2,3,4])

# ── Universal helpers ─────────────────────────────────────────
def to_ft(x):
    if isinstance(x, torch.Tensor): return x.float()
    if hasattr(x, 'tolist'): return torch.tensor(x.tolist(), dtype=torch.float32)
    if isinstance(x, list):  return torch.tensor(x, dtype=torch.float32)
    return torch.tensor([x], dtype=torch.float32)

def to_list(x):
    if isinstance(x, torch.Tensor): return x.detach().cpu().float().tolist()
    if hasattr(x, 'tolist'): return x.tolist()
    return list(x)

def load_npy_as_tensor(path):
    with open(path,'rb') as f:
        f.read(6); f.read(2)
        hdr_len = struct.unpack('<H', f.read(2))[0]
        hdr     = f.read(hdr_len).decode('latin1')
        shape_m = re.search(r"'shape':\s*\(([^)]+)\)", hdr)
        shape   = tuple(int(s.strip()) for s in
                        shape_m.group(1).split(',') if s.strip())
        dtype_m = re.search(r"'descr':\s*'([^']+)'", hdr)
        ds      = dtype_m.group(1).replace('<','').replace('>','').replace('=','')
        fmt, sz = ('d',8) if ds=='f8' else ('f',4)
        n = 1
        for s in shape: n *= s
        data = list(struct.unpack(f'{n}{fmt}', f.read(n*sz)))
    return torch.tensor(data, dtype=torch.float32).reshape(shape)

def haversine_torch(lat1, lon1, lat2, lon2):
    R=6371.0
    la1=torch.deg2rad(to_ft(lat1)); la2=torch.deg2rad(to_ft(lat2))
    lo1=torch.deg2rad(to_ft(lon1)); lo2=torch.deg2rad(to_ft(lon2))
    dlat=la2-la1; dlon=lo2-lo1
    a=(dlat/2).sin()**2+la1.cos()*la2.cos()*(dlon/2).sin()**2
    return R*2*torch.arcsin(a.sqrt().clamp(0,1))

# ── Core metric functions ─────────────────────────────────────
def clf_metrics(y_true, y_prob, thr=0.5):
    yt=to_ft(y_true); yp=to_ft(y_prob)
    pred=(yp>=thr).float()
    tp=((pred==1)&(yt==1)).sum().float()
    fp=((pred==1)&(yt==0)).sum().float()
    fn=((pred==0)&(yt==1)).sum().float()
    tn=((pred==0)&(yt==0)).sum().float()
    prec=float(tp/(tp+fp).clamp(min=1))
    rec =float(tp/(tp+fn).clamp(min=1))
    f1  =float(2*prec*rec/max(prec+rec,1e-8))
    acc =float((tp+tn)/len(yt))
    n_pos=int((yt==1).sum()); n_neg=int((yt==0).sum())
    if n_pos==0 or n_neg==0: auc=0.5; prauc=float(prec)
    else:
        sidx=torch.argsort(yp)
        rank=torch.arange(1,len(yt)+1,dtype=torch.float32)
        rs  =rank[torch.argsort(yp)][yt[torch.argsort(yp)]==1].sum()
        auc =float(((rs-n_pos*(n_pos+1)/2)/(n_pos*n_neg)).clamp(0,1))
        sidx2=torch.argsort(yp,descending=True); ys=yt[sidx2]
        tc=torch.cumsum(ys,0)
        pk=tc/torch.arange(1,len(yt)+1,dtype=torch.float32)
        rk=tc/max(n_pos,1)
        rd=torch.diff(rk,prepend=torch.zeros(1))
        prauc=float((pk*rd).sum().clamp(0,1))
    return {'acc':acc,'prec':prec,'recall':rec,'f1':f1,'auc':auc,'prauc':prauc}

def reg_metrics(y_true, y_pred, mask=None):
    yt=to_ft(y_true); yp=to_ft(y_pred)
    if mask is not None:
        m=to_ft(mask).bool()
        if m.sum()==0: return {'rmse':0.0,'mae':0.0}
        yt=yt[m]; yp=yp[m]
    return {'rmse':float(((yp-yt)**2).mean().sqrt()),
            'mae': float((yp-yt).abs().mean())}

def roc_curve_torch(y_true, y_prob, n_thresh=100):
    yt=to_ft(y_true); yp=to_ft(y_prob)
    thrs=torch.linspace(0,1,n_thresh); fprs,tprs=[],[]
    for t in thrs:
        pred=(yp>=t).float()
        tp=((pred==1)&(yt==1)).sum().float()
        fp=((pred==1)&(yt==0)).sum().float()
        fn=((pred==0)&(yt==1)).sum().float()
        tn=((pred==0)&(yt==0)).sum().float()
        fprs.append(float(fp/(fp+tn).clamp(min=1)))
        tprs.append(float(tp/(tp+fn).clamp(min=1)))
    return fprs,tprs

def pr_curve_torch(y_true, y_prob, n_thresh=100):
    yt=to_ft(y_true); yp=to_ft(y_prob)
    thrs=torch.linspace(0,1,n_thresh); precs,recs=[],[]
    for t in thrs:
        pred=(yp>=t).float()
        tp=((pred==1)&(yt==1)).sum().float()
        fp=((pred==1)&(yt==0)).sum().float()
        fn=((pred==0)&(yt==1)).sum().float()
        precs.append(float(tp/(tp+fp).clamp(min=1)))
        recs.append(float(tp/(tp+fn).clamp(min=1)))
    return recs,precs

# ── Load metadata ─────────────────────────────────────────────
print("Loading all results...")
with open(f"{OUT_DIR}dataset_meta.pkl",'rb') as f: meta=pickle.load(f)
with open(f"{OUT_DIR}model_config.pkl",'rb') as f: cfg=pickle.load(f)
with open(f"{OUT_DIR}train_history.pkl",'rb') as f: history=pickle.load(f)
with open(f"{BASELINE_DIR}all_baselines.pkl",'rb') as f: bl=pickle.load(f)

chunk_index=meta['chunk_index']; N_common=meta['N_common']
n_train=meta['n_train'];         n_val=meta['n_val']
pos_weights=meta['pos_weights']; label_norm=meta['label_norm']
STEP=meta.get('step',20)

Y_test_t=to_ft(bl['Y_test'])

def cuda_works():
    if not torch.cuda.is_available(): return False
    try:
        a=torch.ones(4,4).cuda(); _=a@a; del a,_
        torch.cuda.empty_cache(); return True
    except: return False
DEVICE=torch.device('cuda' if cuda_works() else 'cpu')
print(f"Device : {DEVICE}")


# MODEL DEFINITIONS
class ETASPriorEmbedding(nn.Module):
    def __init__(self,n_etas=5,embed_dim=32):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(n_etas,64),nn.LayerNorm(64),
            nn.GELU(),nn.Linear(64,embed_dim),nn.GELU())
        self.idx=[6,7,8,9,10]
    def forward(self,x): return self.net(x[:,self.idx])

class STGATBlock(nn.Module):
    def __init__(self,in_dim,out_dim,n_heads=4,edge_dim=4,dropout=0.1):
        super().__init__()
        self.gat=GATv2Conv(in_dim,out_dim//n_heads,heads=n_heads,
                           edge_dim=edge_dim,concat=True,dropout=dropout)
        self.norm=nn.LayerNorm(out_dim); self.drop=nn.Dropout(dropout)
        self.res=nn.Linear(in_dim,out_dim) if in_dim!=out_dim else nn.Identity()
    def forward(self,x,ei,ea):
        return self.norm(self.drop(self.gat(x,ei,ea))+self.res(x))

class MultiWindowGNNEncoder(nn.Module):
    def __init__(self,in_dim,hidden_dim,out_dim,n_layers=3,n_heads=4,
                 edge_dim=4,etas_embed_dim=32,dropout=0.1,
                 use_etas=True,use_gnn=True,n_windows=3):
        super().__init__()
        self.use_etas=use_etas; self.use_gnn=use_gnn; self.n_windows=n_windows
        if use_etas:
            self.etas=ETASPriorEmbedding(5,etas_embed_dim)
            proj_in=in_dim+etas_embed_dim
        else:
            proj_in=in_dim
        self.in_proj=nn.Linear(proj_in,hidden_dim)
        if use_gnn:
            self.layers=nn.ModuleList([STGATBlock(hidden_dim,hidden_dim,
                n_heads,edge_dim,dropout) for _ in range(n_layers)])
        self.pool_proj=nn.Linear(hidden_dim*2,out_dim)
        self.fuse=nn.Sequential(nn.Linear(out_dim*n_windows,out_dim*2),
            nn.LayerNorm(out_dim*2),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(out_dim*2,out_dim))
    def encode(self,data):
        x,ei,ea,b=data.x,data.edge_index,data.edge_attr,data.batch
        if self.use_etas:
            x=self.in_proj(torch.cat([x,self.etas(x)],dim=1))
        else:
            x=self.in_proj(x)
        if self.use_gnn:
            for l in self.layers: x=l(x,ei,ea)
        gm=global_mean_pool(x,b); gx=global_max_pool(x,b)
        return self.pool_proj(torch.cat([gm,gx],dim=1))
    def forward(self,*graphs):
        encs=[self.encode(g) for g in graphs]
        return self.fuse(torch.cat(encs,dim=1))

class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=512,dropout=0.1):
        super().__init__()
        self.drop=nn.Dropout(dropout)
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(0,max_len).unsqueeze(1).float()
        div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x): return self.drop(x+self.pe[:,:x.size(1)])

class TemporalTransformerEncoder(nn.Module):
    def __init__(self,d_model,n_heads=8,n_layers=4,ffn_dim=512,
                 dropout=0.1,n_horizons=5,use_transformer=True):
        super().__init__()
        self.use_transformer=use_transformer
        self.pos=PositionalEncoding(d_model,dropout=dropout)
        self.htok=nn.Parameter(torch.randn(n_horizons,d_model)*0.02)
        if use_transformer:
            self.tf=nn.TransformerEncoder(nn.TransformerEncoderLayer(
                d_model,n_heads,ffn_dim,dropout,activation='gelu',
                batch_first=True,norm_first=True),
                num_layers=n_layers,norm=nn.LayerNorm(d_model))
        else:
            self.proj=nn.Linear(d_model,d_model*n_horizons)
        self.n_horizons=n_horizons; self.d_model=d_model
    def forward(self,x):
        B=x.size(0)
        if self.use_transformer:
            seq=torch.cat([x.unsqueeze(1),
                           self.htok.unsqueeze(0).expand(B,-1,-1)],dim=1)
            return self.tf(self.pos(seq))[:,1:,:]
        else:
            out=self.proj(x).view(B,self.n_horizons,self.d_model)
            return out

class HorizonHead(nn.Module):
    def __init__(self,d_model,hidden=128,dropout=0.1):
        super().__init__()
        def mlp(o): return nn.Sequential(nn.Linear(d_model,hidden),
            nn.LayerNorm(hidden),nn.GELU(),nn.Dropout(dropout),nn.Linear(hidden,o))
        self.occ=mlp(1); self.mag=mlp(1); self.loc=mlp(2)
    def forward(self,x): return self.occ(x),self.mag(x),self.loc(x)

class AfterShockNet(nn.Module):
    def __init__(self,n_node_feat=16,n_edge_feat=4,gnn_hidden=128,gnn_out=256,
                 gnn_layers=3,gnn_heads=4,etas_embed=32,tf_d_model=256,tf_heads=8,
                 tf_layers=4,tf_ffn=512,head_hidden=128,dropout=0.1,n_horizons=5,
                 use_etas=True,use_gnn=True,use_transformer=True,n_windows=3):
        super().__init__()
        self.n_windows=n_windows
        self.gnn=MultiWindowGNNEncoder(n_node_feat,gnn_hidden,gnn_out,
            gnn_layers,gnn_heads,n_edge_feat,etas_embed,dropout,
            use_etas=use_etas,use_gnn=use_gnn,n_windows=n_windows)
        self.proj=(nn.Linear(gnn_out,tf_d_model)
                   if gnn_out!=tf_d_model else nn.Identity())
        self.tf=TemporalTransformerEncoder(tf_d_model,tf_heads,tf_layers,
                                            tf_ffn,dropout,n_horizons,
                                            use_transformer=use_transformer)
        self.heads=nn.ModuleList([HorizonHead(tf_d_model,head_hidden,dropout)
                                  for _ in range(n_horizons)])
        self.horizons=HORIZONS
    def forward(self,*graphs):
        h=self.tf(self.proj(self.gnn(*graphs))); out={}
        for i,hn in enumerate(self.horizons):
            o,m,l=self.heads[i](h[:,i,:]); out[hn]={'occ_logit':o,'mag':m,'loc':l}
        return out

# ── Load full AfterShockNet ───────────────────────────────────
def load_asn(use_etas=True, use_gnn=True, use_transformer=True, n_windows=3,
             checkpoint='best_model.pt'):
    m=AfterShockNet(**{k:cfg[k] for k in [
        'n_node_feat','n_edge_feat','gnn_hidden','gnn_out','gnn_layers','gnn_heads',
        'etas_embed','tf_d_model','tf_heads','tf_layers','tf_ffn','head_hidden',
        'dropout','n_horizons']},
        use_etas=use_etas,use_gnn=use_gnn,
        use_transformer=use_transformer,n_windows=n_windows).to(DEVICE)
    ckpt=torch.load(f"{OUT_DIR}{checkpoint}",weights_only=False,map_location=DEVICE)
    try: m.load_state_dict(ckpt['model_state'],strict=False)
    except: pass
    m.eval(); return m

model=load_asn()
ckpt=torch.load(f"{OUT_DIR}best_model.pt",weights_only=False,map_location=DEVICE)
print(f"AfterShockNet loaded — epoch {ckpt['epoch']}  val={ckpt['val_loss']:.4f}")

# ── Index maps and Dataset ────────────────────────────────────
def build_index_map(w_name, chunk_index, n_common):
    idx=[]
    for entry in chunk_index[w_name]:
        for li in range(entry['end']-entry['start']):
            idx.append((entry['path'],entry['start']+li))
            if len(idx)>=n_common: return idx
    return idx

idx24=build_index_map('w24',chunk_index,N_common)
idx48=build_index_map('w48',chunk_index,N_common)
idx72=build_index_map('w72',chunk_index,N_common)

class ChunkStreamDataset(Dataset):
    def __init__(self,m24,m48,m72,horizons,indices,n_windows=3):
        self.maps=[m24,m48,m72][:n_windows]; self.n_windows=n_windows
        self.horizons=horizons; self.indices=indices
        self._caches=[{} for _ in range(n_windows)]
    def _get(self,ci,path,li):
        if path not in self._caches[ci]:
            self._caches[ci].clear(); gc.collect()
            self._caches[ci][path]=torch.load(path,weights_only=False)
        return self._caches[ci][path][li]
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        gi=self.indices[i]
        graphs=[self._get(ci,*self.maps[ci][gi]) for ci in range(self.n_windows)]
        parts=[]
        for h in self.horizons:
            parts+=[getattr(graphs[0],f'{h}_occ'),getattr(graphs[0],f'{h}_mag'),
                    getattr(graphs[0],f'{h}_lat'),getattr(graphs[0],f'{h}_lon')]
        return (*graphs, torch.cat(parts,dim=0))

def collate_fn(batch):
    *graph_lists, lbls = zip(*batch)
    batched=[Batch.from_data_list(list(gl)) for gl in graph_lists]
    return (*batched, torch.stack(lbls,dim=0))

def make_loader(indices, n_windows=3, batch_size=32):
    ds=ChunkStreamDataset(idx24,idx48,idx72,HORIZONS,indices,n_windows)
    return DataLoader(ds,batch_size=batch_size,shuffle=False,
                      collate_fn=collate_fn,num_workers=0)

test_indices=list(range(n_train+n_val,N_common))

# ── Inference function ────────────────────────────────────────
def run_inference(model, loader, n_windows=3):
    preds={h:{'occ_prob':[],'mag':[],'loc':[]} for h in HORIZONS}
    labels=[]
    with torch.no_grad():
        for batch in loader:
            *graphs, lbl = batch
            graphs=[g.to(DEVICE) for g in graphs]
            out=model(*graphs[:n_windows])
            labels.append(lbl.float().cpu())
            for h in HORIZONS:
                preds[h]['occ_prob'].append(
                    torch.sigmoid(out[h]['occ_logit']).squeeze(1).float().cpu())
                preds[h]['mag'].append(out[h]['mag'].squeeze(1).float().cpu())
                preds[h]['loc'].append(out[h]['loc'].float().cpu())
            del graphs,out
            if DEVICE.type=='cuda': torch.cuda.empty_cache()
    labels_t=torch.cat(labels,dim=0)
    for h in HORIZONS:
        preds[h]['occ_prob']=torch.cat(preds[h]['occ_prob'])
        preds[h]['mag']=torch.cat(preds[h]['mag'])
        preds[h]['loc']=torch.cat(preds[h]['loc'])
    gc.collect()
    return preds,labels_t

def compute_all_metrics(preds, labels_t):
    results={}
    for i,h in enumerate(HORIZONS):
        base=i*4
        occ_lbl=labels_t[:,base].float()
        mag_lbl=labels_t[:,base+1].float()
        lat_lbl=labels_t[:,base+2].float()
        lon_lbl=labels_t[:,base+3].float()
        occ_prob=preds[h]['occ_prob']
        mag_pred=preds[h]['mag']
        loc_pred=preds[h]['loc']
        mask=occ_lbl==1
        cm=clf_metrics(occ_lbl,occ_prob)
        rm=reg_metrics(mag_lbl,mag_pred,mask=mask)
        if mask.sum()>0:
            hav=haversine_torch(lat_lbl[mask],lon_lbl[mask],
                                loc_pred[mask,0],loc_pred[mask,1])
            sp={'mean_km':float(hav.mean()),'median_km':float(hav.median()),
                'p90_km':float(hav.quantile(0.9))}
        else: sp={'mean_km':0.0,'median_km':0.0,'p90_km':0.0}
        results[h]={**cm,**rm,**sp,'occ_lbl':occ_lbl,'occ_prob':occ_prob,
                    'mag_lbl':mag_lbl,'mag_pred':mag_pred,
                    'lat_lbl':lat_lbl,'lon_lbl':lon_lbl,'loc_pred':loc_pred}
    return results

# ── 6.1 Main inference ────────────────────────────────────────
print("\n6.1 Running AfterShockNet inference...")
test_loader=make_loader(test_indices,n_windows=3,batch_size=32)
asn_preds,asn_labels_t=run_inference(model,test_loader,n_windows=3)
asn_results=compute_all_metrics(asn_preds,asn_labels_t)
for h in HORIZONS:
    r=asn_results[h]
    print(f"  ASN {h:>4s}  F1={r['f1']:.4f}  AUC={r['auc']:.4f}  "
          f"PR-AUC={r['prauc']:.4f}  MagRMSE={r['rmse']:.4f}  "
          f"SpatMean={r['mean_km']:.1f}km")

# ── 6.2 Spatial baselines ─────────────────────────────────────
print("\n6.2 Spatial metrics for baselines...")
eq_arr_raw_t=load_npy_as_tensor(f"{OUT_DIR}eq_arr_raw.npy")
anchor_all=list(range(5,eq_arr_raw_t.shape[0],STEP))[:N_common]
test_anchors=anchor_all[n_train+n_val:]
lat_anchor=eq_arr_raw_t[test_anchors,1].float()
lon_anchor=eq_arr_raw_t[test_anchors,2].float()
del eq_arr_raw_t; gc.collect()

bl_spatial={}
for bl_name in ['ETAS','XGBoost','LSTM','Transformer']:
    bl_spatial[bl_name]={}
    for i,h in enumerate(HORIZONS):
        base=i*4
        occ_lbl=Y_test_t[:,base].float()
        lat_true=Y_test_t[:,base+2].float()
        lon_true=Y_test_t[:,base+3].float()
        mask=occ_lbl.bool()
        if mask.sum()==0:
            bl_spatial[bl_name][h]={'mean_km':0.0,'median_km':0.0,'p90_km':0.0}; continue
        hav=haversine_torch(lat_true[mask],lon_true[mask],
                            lat_anchor[mask],lon_anchor[mask])
        bl_spatial[bl_name][h]={'mean_km':float(hav.mean()),
                                 'median_km':float(hav.median()),
                                 'p90_km':float(hav.quantile(0.9))}

# ── 6.3 Results table ─────────────────────────────────────────
print("\n6.3 Building comparison table...")
all_results={'ETAS':bl['etas_results'],'XGBoost':bl['xgb_results'],
             'LSTM':bl['lstm_results'],'Transformer':bl['tf_results'],
             'AfterShockNet':asn_results}

def get_metric(mn,h,mk):
    if   mn=='AfterShockNet': return asn_results[h].get(mk,0)
    elif mn=='ETAS':          return bl['etas_results'][h].get(mk,0)
    elif mn=='XGBoost':       return bl['xgb_results'][h].get(mk,0)
    elif mn=='LSTM':          return bl['lstm_results'][h].get(mk,0)
    else:                     return bl['tf_results'][h].get(mk,0)

rows=[]
for mn,res in all_results.items():
    for h in HORIZONS:
        r=res[h]
        sp=asn_results[h] if mn=='AfterShockNet' else bl_spatial[mn][h]
        rows.append({'Model':mn,'Horizon':h,
            'Accuracy':     round(r.get('acc',0),4),
            'Precision':    round(r.get('prec',r.get('precision',0)),4),
            'Recall':       round(r.get('recall',0),4),
            'F1':           round(r.get('f1',0),4),
            'AUC-ROC':      round(r.get('auc',0),4),
            'PR-AUC':       round(r.get('prauc',0),4),
            'Mag-RMSE':     round(r.get('rmse',0),4),
            'Mag-MAE':      round(r.get('mae',0),4),
            'Spat-Mean-km': round(sp.get('mean_km',0),2),
            'Spat-Med-km':  round(sp.get('median_km',0),2),
            'Spat-P90-km':  round(sp.get('p90_km',0),2)})
results_df=pd.DataFrame(rows)
results_df.to_csv(f"{OUT_DIR}evaluation_results.csv",index=False)
print(results_df.to_string(index=False))


# SECTION A: STATISTICAL SIGNIFICANCE TESTING
# Paired bootstrap permutation test — pure torch, no scipy
print("\n" + "="*65)
print("A. STATISTICAL SIGNIFICANCE TESTING")
print("="*65)

def bootstrap_f1_diff(y_true, prob_a, prob_b, n_boot=2000, thr=0.5, seed=42):
    """
    Paired bootstrap test: H0: F1(A) - F1(B) = 0
    Returns observed_diff, p_value, ci_low, ci_high
    Pure Python/torch — no scipy/numpy required.
    """
    torch.manual_seed(seed)
    yt=to_ft(y_true); pa=to_ft(prob_a); pb=to_ft(prob_b)
    N=len(yt)

    def f1_from_tensors(y,p):
        pred=(p>=thr).float()
        tp=((pred==1)&(y==1)).sum().float()
        fp=((pred==1)&(y==0)).sum().float()
        fn=((pred==0)&(y==1)).sum().float()
        prec=float(tp/(tp+fp).clamp(min=1))
        rec =float(tp/(tp+fn).clamp(min=1))
        return float(2*prec*rec/max(prec+rec,1e-8))

    obs_diff=f1_from_tensors(yt,pa)-f1_from_tensors(yt,pb)
    boot_diffs=[]
    for _ in range(n_boot):
        idx=torch.randint(0,N,(N,))
        boot_diffs.append(f1_from_tensors(yt[idx],pa[idx])-f1_from_tensors(yt[idx],pb[idx]))
    boot_diffs_t=torch.tensor(boot_diffs)
    # two-sided p-value: proportion of bootstrap diffs more extreme than observed
    p_val=float((boot_diffs_t.abs()>=abs(obs_diff)).float().mean())
    # 95% CI
    sorted_diffs=sorted(boot_diffs)
    ci_low =sorted_diffs[int(0.025*n_boot)]
    ci_high=sorted_diffs[int(0.975*n_boot)]
    return obs_diff, p_val, ci_low, ci_high

def wilcoxon_signed_rank(diffs):
    """
    Pure Python Wilcoxon signed-rank statistic and approximate p-value.
    diffs: list of per-sample differences (A_i - B_i).
    Returns W_stat, p_approx.
    """
    nonzero=[d for d in diffs if abs(d)>1e-10]
    if len(nonzero)==0: return 0.0, 1.0
    n=len(nonzero)
    abs_diffs=sorted([(abs(d),i,d) for i,d in enumerate(nonzero)])
    # assign ranks (1-based)
    ranks=[0.0]*n
    i=0
    while i<n:
        j=i
        while j<n and abs(abs_diffs[j][0]-abs_diffs[i][0])<1e-10: j+=1
        avg_rank=(i+1+j)/2.0
        for k in range(i,j): ranks[k]=avg_rank
        i=j
    W_plus=sum(r for r,(_,_,d) in zip(ranks,abs_diffs) if d>0)
    W_minus=sum(r for r,(_,_,d) in zip(ranks,abs_diffs) if d<0)
    W=min(W_plus,W_minus)
    # Normal approximation
    mu=n*(n+1)/4.0
    sigma=math.sqrt(n*(n+1)*(2*n+1)/24.0)
    if sigma<1e-10: return W,1.0
    z=(W-mu)/sigma
    # two-tailed p from standard normal CDF approximation
    t=abs(z)
    p_approx=2*(1-0.5*(1+math.erf(t/math.sqrt(2))))
    return W, p_approx

print("\nPaired Bootstrap F1 Significance: AfterShockNet vs each baseline")
print(f"{'Baseline':<14} {'Horizon':<8} {'ObsDiff':>9} {'p-value':>9} "
      f"{'CI_low':>9} {'CI_high':>9} {'Sig':>5}")
print("-"*65)

sig_rows=[]
comparisons=[('LSTM',bl['lstm_preds']),
             ('Transformer',bl['tf_preds']),
             ('XGBoost',bl['xgb_preds']),
             ('ETAS',bl['etas_preds'])]

for bl_name,bl_pr in comparisons:
    for h in HORIZONS:
        base=HORIZONS.index(h)*4
        y_true=asn_labels_t[:,base].float()
        prob_asn=asn_preds[h]['occ_prob']
        prob_bl =to_ft(bl_pr[h]['occ_prob'])
        # align lengths
        n_min=min(len(y_true),len(prob_asn),len(prob_bl))
        y_true=y_true[:n_min]; prob_asn=prob_asn[:n_min]; prob_bl=prob_bl[:n_min]
        obs,pval,ci_lo,ci_hi=bootstrap_f1_diff(y_true,prob_asn,prob_bl,n_boot=1000)
        sig='***' if pval<0.001 else ('**' if pval<0.01 else ('*' if pval<0.05 else 'ns'))
        print(f"{bl_name:<14} {h:<8} {obs:>+9.4f} {pval:>9.4f} "
              f"{ci_lo:>+9.4f} {ci_hi:>+9.4f} {sig:>5}")
        sig_rows.append({'Baseline':bl_name,'Horizon':h,'ObsDiff':round(obs,4),
                         'p_value':round(pval,4),'CI_low':round(ci_lo,4),
                         'CI_high':round(ci_hi,4),'Significance':sig})

sig_df=pd.DataFrame(sig_rows)
sig_df.to_csv(f"{OUT_DIR}statistical_significance.csv",index=False)
print("\nWilcoxon Signed-Rank: per-sample F1 contributions")
print(f"{'Baseline':<14} {'Horizon':<8} {'W_stat':>9} {'p_approx':>10} {'Sig':>5}")
print("-"*55)

wilcox_rows=[]
for bl_name,bl_pr in comparisons:
    for h in HORIZONS:
        base=HORIZONS.index(h)*4
        y_true=asn_labels_t[:,base].float()
        prob_asn=asn_preds[h]['occ_prob']
        prob_bl =to_ft(bl_pr[h]['occ_prob'])
        n_min=min(len(y_true),len(prob_asn),len(prob_bl))
        y_true=y_true[:n_min]
        prob_asn=prob_asn[:n_min]
        prob_bl=prob_bl[:n_min]
        # per-sample indicator: correct ASN vs correct BL
        pred_asn=(prob_asn>=0.5).float()
        pred_bl =(prob_bl>=0.5).float()
        corr_asn=(pred_asn==y_true).float()
        corr_bl =(pred_bl==y_true).float()
        diffs=to_list(corr_asn-corr_bl)
        W,pval=wilcoxon_signed_rank(diffs)
        sig='***' if pval<0.001 else ('**' if pval<0.01 else ('*' if pval<0.05 else 'ns'))
        print(f"{bl_name:<14} {h:<8} {W:>9.1f} {pval:>10.4f} {sig:>5}")
        wilcox_rows.append({'Baseline':bl_name,'Horizon':h,
                            'W_stat':round(W,2),'p_approx':round(pval,4),
                            'Significance':sig})

pd.DataFrame(wilcox_rows).to_csv(f"{OUT_DIR}wilcoxon_results.csv",index=False)

# Figure: Significance heatmap
fig,axes=plt.subplots(1,2,figsize=(18,6))
fig.suptitle("Statistical Significance — AfterShockNet vs Baselines",
             fontsize=14,fontweight='bold')
baselines_list=['ETAS','XGBoost','LSTM','Transformer']

for ax_i,(ax,df,title) in enumerate(zip(
    axes,
    [sig_df,pd.DataFrame(wilcox_rows)],
    ['Bootstrap p-value (paired F1)','Wilcoxon p-value (per-sample accuracy)'])):
    mat=[]
    for bl_n in baselines_list:
        row=[]
        for h in HORIZONS:
            sub=df[(df['Baseline']==bl_n)&(df['Horizon']==h)]
            if len(sub)>0: row.append(float(sub.iloc[0]['p_value' if 'p_value' in df.columns else 'p_approx']))
            else: row.append(1.0)
        mat.append(row)
    im=ax.imshow(mat,cmap='RdYlGn_r',vmin=0,vmax=0.1,aspect='auto')
    ax.set_xticks(range(len(HORIZONS))); ax.set_xticklabels(HORIZONS,fontsize=11)
    ax.set_yticks(range(len(baselines_list))); ax.set_yticklabels(baselines_list,fontsize=11)
    ax.set_title(title,fontsize=11,fontweight='bold')
    plt.colorbar(im,ax=ax,shrink=0.8,label='p-value')
    for i in range(len(baselines_list)):
        for j in range(len(HORIZONS)):
            pv=mat[i][j]
            sig='***' if pv<0.001 else ('**' if pv<0.01 else ('*' if pv<0.05 else 'ns'))
            ax.text(j,i,f'{pv:.3f}\n{sig}',ha='center',va='center',
                    fontsize=8,fontweight='bold',color='black')
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig16_significance_heatmap.png",dpi=150,bbox_inches='tight')
plt.close(); print("\n  Saved fig16_significance_heatmap.png")


# SECTION B: ABLATION STUDY
# Four variants tested on the EXISTING best_model.pt weights
# with architectural masking — no retraining required
print("\n" + "="*65)
print("B. ABLATION STUDY")
print("="*65)

ablation_configs = {
    'Full AfterShockNet':
        {'use_etas':True,  'use_gnn':True,  'use_transformer':True,  'n_windows':3},
    'w/o ETAS Prior':
        {'use_etas':False, 'use_gnn':True,  'use_transformer':True,  'n_windows':3},
    'w/o GNN Encoder':
        {'use_etas':True,  'use_gnn':False, 'use_transformer':True,  'n_windows':3},
    'w/o Transformer':
        {'use_etas':True,  'use_gnn':True,  'use_transformer':False,  'n_windows':3},
    'Single Window (w24 only)':
        {'use_etas':True,  'use_gnn':True,  'use_transformer':True,  'n_windows':1},
}

ABLATION_COLORS={
    'Full AfterShockNet':   '#E53935',
    'w/o ETAS Prior':       '#FB8C00',
    'w/o GNN Encoder':      '#43A047',
    'w/o Transformer':      '#1E88E5',
    'Single Window (w24 only)':'#8E24AA',
}

ablation_results={}
for variant_name, vcfg in ablation_configs.items():
    print(f"\n  Running: {variant_name}")
    nw=vcfg['n_windows']
    vmodel=AfterShockNet(**{k:cfg[k] for k in [
        'n_node_feat','n_edge_feat','gnn_hidden','gnn_out','gnn_layers','gnn_heads',
        'etas_embed','tf_d_model','tf_heads','tf_layers','tf_ffn','head_hidden',
        'dropout','n_horizons']},
        use_etas=vcfg['use_etas'],use_gnn=vcfg['use_gnn'],
        use_transformer=vcfg['use_transformer'],n_windows=nw).to(DEVICE)
    # Load weights with strict=False so mismatched layers are randomly init
    ckpt_w=torch.load(f"{OUT_DIR}best_model.pt",weights_only=False,map_location=DEVICE)
    # Filter out keys with shape mismatch — load only compatible weights
    model_state = vmodel.state_dict()
    filtered_ckpt = {}
    skipped = []
    for k, v in ckpt_w['model_state'].items():
        if k in model_state and model_state[k].shape == v.shape:
            filtered_ckpt[k] = v
        else:
            skipped.append(k)
    model_state.update(filtered_ckpt)
    vmodel.load_state_dict(model_state)
    if skipped:
        print(f"    Skipped {len(skipped)} mismatched keys: {skipped[:3]}{'...' if len(skipped)>3 else ''}")
    vmodel.eval()
    
    vloader=make_loader(test_indices,n_windows=nw,batch_size=32)
    vpreds,vlabels=run_inference(vmodel,vloader,n_windows=nw)
    vres=compute_all_metrics(vpreds,vlabels)
    ablation_results[variant_name]=vres
    for h in HORIZONS:
        r=vres[h]
        print(f"    {h}: F1={r['f1']:.4f}  PR-AUC={r['prauc']:.4f}  "
              f"MagRMSE={r['rmse']:.4f}")
    del vmodel; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()

# Save ablation table
ab_rows=[]
for vname,vres in ablation_results.items():
    for h in HORIZONS:
        r=vres[h]
        ab_rows.append({'Variant':vname,'Horizon':h,
                        'F1':round(r['f1'],4),'PR-AUC':round(r['prauc'],4),
                        'AUC-ROC':round(r['auc'],4),'MagRMSE':round(r['rmse'],4),
                        'SpatMean':round(r.get('mean_km',0),2)})
ab_df=pd.DataFrame(ab_rows)
ab_df.to_csv(f"{OUT_DIR}ablation_results.csv",index=False)

# Figure: Ablation F1 line plot per horizon
fig,axes=plt.subplots(1,3,figsize=(21,6))
fig.suptitle("Ablation Study — Component Contribution Analysis",
             fontsize=14,fontweight='bold')
metrics_ab=[('f1','F1 Score'),('prauc','PR-AUC'),('rmse','Magnitude RMSE')]
for ax,(mk,ylabel) in zip(axes,metrics_ab):
    for vname,vres in ablation_results.items():
        vals=[vres[h].get(mk,0) for h in HORIZONS]
        lw=3.0 if 'Full' in vname else 1.8
        ls='-'  if 'Full' in vname else '--'
        ax.plot(HORIZONS,vals,marker='o',linewidth=lw,linestyle=ls,
                color=ABLATION_COLORS[vname],label=vname,markersize=6)
    ax.set_xlabel('Forecast Horizon',fontsize=12)
    ax.set_ylabel(ylabel,fontsize=12)
    ax.set_title(ylabel,fontsize=12,fontweight='bold')
    ax.legend(fontsize=8,loc='best'); ax.grid(True,alpha=0.3)
    if mk!='rmse': ax.set_ylim(0,1.05)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig17_ablation_study.png",dpi=150,bbox_inches='tight')
plt.close(); print("\n  Saved fig17_ablation_study.png")

# Figure: Ablation delta heatmap (drop from full model)
full_res=ablation_results['Full AfterShockNet']
variants_no_full=[v for v in ablation_configs if v!='Full AfterShockNet']
fig,axes=plt.subplots(1,2,figsize=(16,5))
fig.suptitle("Ablation — F1 and PR-AUC Drop from Full AfterShockNet",
             fontsize=13,fontweight='bold')
for ax,mk in zip(axes,['f1','prauc']):
    mat=[[full_res[h].get(mk,0)-ablation_results[v][h].get(mk,0)
          for h in HORIZONS] for v in variants_no_full]
    im=ax.imshow(mat,cmap='Reds',vmin=0,aspect='auto')
    ax.set_xticks(range(len(HORIZONS))); ax.set_xticklabels(HORIZONS,fontsize=11)
    ax.set_yticks(range(len(variants_no_full)))
    ax.set_yticklabels(variants_no_full,fontsize=10)
    ax.set_title(f'{mk.upper()} drop (Full − Variant)',fontsize=11,fontweight='bold')
    plt.colorbar(im,ax=ax,shrink=0.8)
    for i in range(len(variants_no_full)):
        for j in range(len(HORIZONS)):
            ax.text(j,i,f'{mat[i][j]:+.3f}',ha='center',va='center',
                    fontsize=9,fontweight='bold')
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig18_ablation_delta.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig18_ablation_delta.png")


# SECTION C: TECTONIC ZONE STRATIFICATION
# Zones derived from plate boundary distance and lat/lon
print("\n" + "="*65)
print("C. TECTONIC ZONE STRATIFICATION")
print("="*65)

eq_df=pd.read_csv(f"{OUT_DIR}eq_features.csv")
FEATURE_COLS=['latitude','longitude','depth','mag','dist_fault_km','dist_pb_km',
              'etas_lambda','etas_mu','etas_n_prior','etas_max_mag','etas_energy',
              'hour_sin','hour_cos','doy_sin','doy_cos','dt_prev_s']
anchor_all2=list(range(5,len(eq_df),STEP))[:N_common]
test_anc2  =anchor_all2[n_train+n_val:]
feat_df    =eq_df.iloc[test_anc2].reset_index(drop=True)

# Tectonic zone assignment from proximity and geographic rules
# Four zones: Subduction, Intraplate, Transform, Ridge/Rift
def assign_tectonic_zone(row):
    dist_pb  = row['dist_pb_km']
    dist_fault=row['dist_fault_km']
    lat      = row['latitude']
    lon      = row['longitude']
    depth    = row['depth']
    # Subduction: near plate boundary, deep, circum-Pacific or Alpide
    if dist_pb < 150 and depth > 50:
        return 'Subduction'
    # Transform: near plate boundary, shallow, mid-latitude
    if dist_pb < 100 and depth <= 50 and abs(lat) < 60:
        return 'Transform'
    # Ridge/Rift: near plate boundary, very shallow, mid-ocean latitudes
    if dist_pb < 200 and depth <= 30 and (
        (-70<lon<-10 and -60<lat<70) or   # Atlantic
        (20<lon<100 and -40<lat<30)):      # Indian
        return 'Ridge/Rift'
    # Intraplate: far from boundaries
    return 'Intraplate'

print("  Assigning tectonic zones...")
zones=feat_df.apply(assign_tectonic_zone,axis=1).tolist()
zone_names=['Subduction','Transform','Ridge/Rift','Intraplate']
zone_colors={'Subduction':'#E53935','Transform':'#FB8C00',
             'Ridge/Rift':'#1E88E5','Intraplate':'#43A047'}

zone_tensor=torch.tensor([zone_names.index(z) for z in zones],dtype=torch.long)
zone_counts={z:zones.count(z) for z in zone_names}
print(f"  Zone distribution: {zone_counts}")

# Compute metrics per zone for AfterShockNet
tectonic_results={z:{} for z in zone_names}
for z in zone_names:
    zmask=zone_tensor==zone_names.index(z)
    if zmask.sum()<10:
        print(f"  Skipping {z}: only {int(zmask.sum())} samples")
        continue
    for i,h in enumerate(HORIZONS):
        base=i*4
        occ_lbl=asn_labels_t[:,base][zmask]
        occ_prob=asn_preds[h]['occ_prob'][zmask]
        mag_lbl =asn_labels_t[:,base+1][zmask]
        mag_pred=asn_preds[h]['mag'][zmask]
        lat_lbl =asn_labels_t[:,base+2][zmask]
        lon_lbl =asn_labels_t[:,base+3][zmask]
        loc_pred=asn_preds[h]['loc'][zmask]
        pos_mask=occ_lbl==1
        cm=clf_metrics(occ_lbl,occ_prob)
        rm=reg_metrics(mag_lbl,mag_pred,mask=pos_mask)
        if pos_mask.sum()>0:
            hav=haversine_torch(lat_lbl[pos_mask],lon_lbl[pos_mask],
                                loc_pred[pos_mask,0],loc_pred[pos_mask,1])
            sp_mean=float(hav.mean())
        else: sp_mean=0.0
        tectonic_results[z][h]={**cm,**rm,'mean_km':sp_mean,
                                 'n_samples':int(zmask.sum())}
    print(f"  {z:>12} (n={zone_counts[z]:>5}): "
          f"1h F1={tectonic_results[z].get('1h',{}).get('f1',0):.3f}  "
          f"24h F1={tectonic_results[z].get('24h',{}).get('f1',0):.3f}  "
          f"72h F1={tectonic_results[z].get('72h',{}).get('f1',0):.3f}")

# Save tectonic table
tz_rows=[]
for z in zone_names:
    for h in HORIZONS:
        if h in tectonic_results[z]:
            r=tectonic_results[z][h]
            tz_rows.append({'Zone':z,'Horizon':h,'N':r['n_samples'],
                            'F1':round(r['f1'],4),'PR-AUC':round(r['prauc'],4),
                            'AUC-ROC':round(r['auc'],4),'MagRMSE':round(r['rmse'],4),
                            'SpatMean_km':round(r['mean_km'],2)})
pd.DataFrame(tz_rows).to_csv(f"{OUT_DIR}tectonic_zone_results.csv",index=False)

# Figure: Tectonic zone performance
fig,axes=plt.subplots(1,3,figsize=(21,6))
fig.suptitle("AfterShockNet Performance by Tectonic Zone",
             fontsize=14,fontweight='bold')
for ax,(mk,ylabel) in zip(axes,[('f1','F1 Score'),('prauc','PR-AUC'),('rmse','Mag RMSE')]):
    x=list(range(len(HORIZONS))); w=0.2
    n_z=len(zone_names)
    offsets=[(i-(n_z-1)/2)*w for i in range(n_z)]
    for zi,(z,off) in enumerate(zip(zone_names,offsets)):
        if not tectonic_results[z]: continue
        vals=[tectonic_results[z].get(h,{}).get(mk,0) for h in HORIZONS]
        bars=ax.bar([xi+off for xi in x],vals,w,label=f"{z} (n={zone_counts[z]:,})",
                    color=zone_colors[z],edgecolor='black',linewidth=0.6,alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(HORIZONS,fontsize=11)
    ax.set_ylabel(ylabel,fontsize=12); ax.set_xlabel('Forecast Horizon',fontsize=12)
    ax.set_title(ylabel,fontsize=12,fontweight='bold')
    ax.legend(fontsize=8); ax.grid(True,axis='y',alpha=0.3)
    if mk!='rmse': ax.set_ylim(0,1.1)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig19_tectonic_zone_performance.png",dpi=150,bbox_inches='tight')
plt.close(); print("\n  Saved fig19_tectonic_zone_performance.png")

# Figure: Tectonic zone heatmap F1
fig,ax=plt.subplots(figsize=(12,5))
fig.suptitle("F1 Score Heatmap — AfterShockNet by Tectonic Zone × Horizon",
             fontsize=13,fontweight='bold')
mat_tz=[[tectonic_results[z].get(h,{}).get('f1',0) for h in HORIZONS]
        for z in zone_names if tectonic_results[z]]
valid_zones=[z for z in zone_names if tectonic_results[z]]
im=ax.imshow(mat_tz,cmap='RdYlGn',vmin=0,vmax=1,aspect='auto')
ax.set_xticks(range(len(HORIZONS))); ax.set_xticklabels(HORIZONS,fontsize=12)
ax.set_yticks(range(len(valid_zones)))
ax.set_yticklabels([f"{z}\n(n={zone_counts[z]:,})" for z in valid_zones],fontsize=11)
plt.colorbar(im,ax=ax,shrink=0.8,label='F1 Score')
for i in range(len(valid_zones)):
    for j in range(len(HORIZONS)):
        ax.text(j,i,f'{mat_tz[i][j]:.3f}',ha='center',va='center',
                fontsize=10,fontweight='bold',color='black')
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig20_tectonic_f1_heatmap.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig20_tectonic_f1_heatmap.png")


# SECTION D: M>=6 POST-MAINSHOCK SEQUENCE EVALUATION
print("\n" + "="*65)
print("D. M>=6 POST-MAINSHOCK SEQUENCE EVALUATION")
print("="*65)

# Identify test anchors that are M>=6 events
mag_col_idx=FEATURE_COLS.index('mag')
# raw magnitudes from eq_features (standardised) — back-transform
# Use raw eq_arr_raw which has [time,lat,lon,depth,mag]
eq_arr_raw2=load_npy_as_tensor(f"{OUT_DIR}eq_arr_raw.npy")
test_anchor_indices_raw=anchor_all2[n_train+n_val:]
anchor_mags=eq_arr_raw2[test_anchor_indices_raw,4].float()  # col 4 = magnitude
del eq_arr_raw2; gc.collect()

m6_mask=anchor_mags>=6.0
n_m6=int(m6_mask.sum())
print(f"  M>=6 anchor events in test set: {n_m6} / {len(test_indices)}")

if n_m6>=10:
    m6_indices_local=torch.where(m6_mask)[0].tolist()

    # Subset existing predictions to M>=6 anchors
    m6_results={}
    for i,h in enumerate(HORIZONS):
        base=i*4
        occ_lbl =asn_labels_t[:,base][m6_mask]
        occ_prob =asn_preds[h]['occ_prob'][m6_mask]
        mag_lbl  =asn_labels_t[:,base+1][m6_mask]
        mag_pred =asn_preds[h]['mag'][m6_mask]
        lat_lbl  =asn_labels_t[:,base+2][m6_mask]
        lon_lbl  =asn_labels_t[:,base+3][m6_mask]
        loc_pred =asn_preds[h]['loc'][m6_mask]
        pos_mask =occ_lbl==1
        cm=clf_metrics(occ_lbl,occ_prob)
        rm=reg_metrics(mag_lbl,mag_pred,mask=pos_mask)
        if pos_mask.sum()>0:
            hav=haversine_torch(lat_lbl[pos_mask],lon_lbl[pos_mask],
                                loc_pred[pos_mask,0],loc_pred[pos_mask,1])
            sp={'mean_km':float(hav.mean()),'median_km':float(hav.median()),
                'p90_km':float(hav.quantile(0.9))}
        else: sp={'mean_km':0.0,'median_km':0.0,'p90_km':0.0}
        m6_results[h]={**cm,**rm,**sp,'n':n_m6,
                       'occ_lbl':occ_lbl,'occ_prob':occ_prob}
        print(f"  M>=6 {h:>4s}: F1={cm['f1']:.4f}  PR-AUC={cm['prauc']:.4f}  "
              f"MagRMSE={rm['rmse']:.4f}  n_pos={int(pos_mask.sum())}/{n_m6}")

    # Baselines on M>=6 subset
    m6_bl_results={}
    for bl_name,bl_pr in [('ETAS',bl['etas_preds']),('XGBoost',bl['xgb_preds']),
                           ('LSTM',bl['lstm_preds']),('Transformer',bl['tf_preds'])]:
        m6_bl_results[bl_name]={}
        for i,h in enumerate(HORIZONS):
            base=i*4
            occ_lbl=Y_test_t[:,base][m6_mask]
            occ_prob=to_ft(bl_pr[h]['occ_prob'])
            n_min=min(len(occ_lbl),len(occ_prob[m6_mask]))
            occ_prob_sub=occ_prob[m6_mask][:n_min]
            occ_lbl_sub=occ_lbl[:n_min]
            cm=clf_metrics(occ_lbl_sub,occ_prob_sub)
            m6_bl_results[bl_name][h]=cm

    # Save M>=6 table
    m6_rows=[]
    for h in HORIZONS:
        r=m6_results[h]
        m6_rows.append({'Model':'AfterShockNet','Horizon':h,'N_anchors':n_m6,
                        'F1':round(r['f1'],4),'PR-AUC':round(r['prauc'],4),
                        'AUC-ROC':round(r['auc'],4),'MagRMSE':round(r['rmse'],4),
                        'SpatMean_km':round(r['mean_km'],2)})
    for bl_name in ['ETAS','XGBoost','LSTM','Transformer']:
        for h in HORIZONS:
            r=m6_bl_results[bl_name][h]
            m6_rows.append({'Model':bl_name,'Horizon':h,'N_anchors':n_m6,
                            'F1':round(r['f1'],4),'PR-AUC':round(r['prauc'],4),
                            'AUC-ROC':round(r['auc'],4),'MagRMSE':0.0,
                            'SpatMean_km':0.0})
    m6_df=pd.DataFrame(m6_rows)
    m6_df.to_csv(f"{OUT_DIR}m6_sequence_results.csv",index=False)

    # Figure: M>=6 comparison — F1 and PR-AUC
    fig,axes=plt.subplots(1,2,figsize=(16,6))
    fig.suptitle(f"Performance on M≥6 Post-Mainshock Sequences (n={n_m6} anchors)",
                 fontsize=14,fontweight='bold')
    all_m6={'AfterShockNet':m6_results,**{b:m6_bl_results[b]
            for b in ['ETAS','XGBoost','LSTM','Transformer']}}
    x=list(range(len(HORIZONS))); w=0.15
    offsets=[(i-2)*w for i in range(5)]
    for ax,mk,ylabel in zip(axes,['f1','prauc'],['F1 Score','PR-AUC']):
        for (mn,mres),off in zip(all_m6.items(),offsets):
            vals=[mres[h].get(mk,0) for h in HORIZONS]
            lw=2.5 if mn=='AfterShockNet' else 1.5
            ax.plot(HORIZONS,vals,marker='o',linewidth=lw,
                    color=COLORS_MODEL[mn],label=mn,markersize=7)
        ax.set_xlabel('Forecast Horizon',fontsize=12)
        ax.set_ylabel(ylabel,fontsize=12)
        ax.set_title(f'M≥6 Sequences — {ylabel}',fontsize=12,fontweight='bold')
        ax.legend(fontsize=9); ax.grid(True,alpha=0.3)
        if mk!='rmse': ax.set_ylim(0,1.05)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}fig21_m6_sequence_performance.png",dpi=150,bbox_inches='tight')
    plt.close(); print("\n  Saved fig21_m6_sequence_performance.png")

    # Figure: M>=6 vs All-events comparison for AfterShockNet
    fig,axes=plt.subplots(1,3,figsize=(21,6))
    fig.suptitle("AfterShockNet: All Events vs M≥6 Post-Mainshock Sequences",
                 fontsize=14,fontweight='bold')
    for ax,(mk,ylabel) in zip(axes,[('f1','F1 Score'),
                                     ('prauc','PR-AUC'),
                                     ('rmse','Mag RMSE')]):
        all_vals=[asn_results[h].get(mk,0) for h in HORIZONS]
        m6_vals =[m6_results[h].get(mk,0)  for h in HORIZONS]
        ax.plot(HORIZONS,all_vals,marker='o',linewidth=2.5,color='#E53935',
                label='All test events',markersize=8)
        ax.plot(HORIZONS,m6_vals,marker='s',linewidth=2.5,color='#1565C0',
                linestyle='--',label='M≥6 anchors only',markersize=8)
        # shaded diff
        for xi,(av,mv) in enumerate(zip(all_vals,m6_vals)):
            col='#C8E6C9' if mv>=av else '#FFCDD2'
            ax.bar(xi,abs(mv-av),bottom=min(av,mv),width=0.3,
                   color=col,alpha=0.4,zorder=0)
        ax.set_xlabel('Forecast Horizon',fontsize=12)
        ax.set_ylabel(ylabel,fontsize=12)
        ax.set_title(ylabel,fontsize=12,fontweight='bold')
        ax.legend(fontsize=10); ax.grid(True,alpha=0.3)
        if mk!='rmse': ax.set_ylim(0,1.05)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}fig22_m6_vs_all_comparison.png",dpi=150,bbox_inches='tight')
    plt.close(); print("  Saved fig22_m6_vs_all_comparison.png")

    # M>=6 magnitude distribution of anchor events
    fig,ax=plt.subplots(figsize=(10,5))
    fig.suptitle("Magnitude Distribution of M≥6 Anchor Events in Test Set",
                 fontsize=13,fontweight='bold')
    mag_vals=to_list(anchor_mags[m6_mask])
    bins=[6.0,6.5,7.0,7.5,8.0,8.5,9.5]
    ax.hist(mag_vals,bins=bins,color='#E53935',edgecolor='black',
            linewidth=0.8,alpha=0.8)
    ax.set_xlabel('Mainshock Magnitude',fontsize=12)
    ax.set_ylabel('Count',fontsize=12)
    ax.set_title(f'n={n_m6} M≥6 anchor events',fontsize=11)
    ax.grid(True,axis='y',alpha=0.3)
    for b_lo,b_hi in zip(bins[:-1],bins[1:]):
        c=sum(1 for v in mag_vals if b_lo<=v<b_hi)
        if c>0: ax.text((b_lo+b_hi)/2,c+0.2,str(c),ha='center',va='bottom',fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}fig23_m6_magnitude_dist.png",dpi=150,bbox_inches='tight')
    plt.close(); print("  Saved fig23_m6_magnitude_dist.png")
else:
    print(f"  Warning: only {n_m6} M>=6 anchors found. Skipping M>=6 figures.")


# ORIGINAL FIGURES 1-15 (unchanged from Step 6)
print("\nGenerating original figures (fig01-fig15)...")

# fig01 — Methodology
HORIZONS_PLOT=HORIZONS
fig=plt.figure(figsize=(22,14)); fig.patch.set_facecolor('#FAFAFA')
ax=fig.add_axes([0,0,1,1]); ax.set_xlim(0,22); ax.set_ylim(0,14); ax.axis('off')
def box(ax,x,y,w,h,color,label,sublabel='',fs=11,rad=0.3,tc='white'):
    from matplotlib.patches import FancyBboxPatch
    p=mpatches.FancyBboxPatch((x,y),w,h,
        boxstyle=f"round,pad=0.1,rounding_size={rad}",
        facecolor=color,edgecolor='white',linewidth=2,zorder=3)
    ax.add_patch(p); cy=y+h/2
    if sublabel:
        ax.text(x+w/2,cy+0.18,label,ha='center',va='center',fontsize=fs,
                fontweight='bold',color=tc,zorder=4)
        ax.text(x+w/2,cy-0.22,sublabel,ha='center',va='center',fontsize=fs-2,
                color=tc,zorder=4,style='italic')
    else:
        ax.text(x+w/2,cy,label,ha='center',va='center',fontsize=fs,
                fontweight='bold',color=tc,zorder=4)
def arr(ax,x1,y1,x2,y2):
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),
        arrowprops=dict(arrowstyle='->',color='#555',lw=2.0),zorder=5)
ax.text(11,13.4,'AfterShockNet: Hybrid ST-GNN + Transformer + ETAS\n'
        'Global Real-Time Aftershock Forecasting Framework',
        ha='center',va='center',fontsize=15,fontweight='bold',color='#1a1a2e')
ax.text(11,12.55,'INPUT DATA SOURCES',ha='center',va='center',
        fontsize=10,color='#555',style='italic')
for sx,sl,sc in zip([0.5,5.5,10.5,15.5],
    ['USGS Earthquake\nCatalog\n(1,037,060 events)','GEM Active\nFaults\n(13,696 faults)',
     'Plate Boundaries\n(6,048 points)','Seismic Hazard\nMap (GSHAP)'],
    ['#2196F3','#4CAF50','#FF9800','#9C27B0']):
    box(ax,sx,11.1,4.2,1.2,sc,sl,fs=9,rad=0.25)
ax.text(11,10.55,'FEATURE ENGINEERING',ha='center',va='center',
        fontsize=10,color='#555',style='italic')
for px,pl,pc in zip([0.5,5.5,10.5,15.5],
    ['ETAS Features\n(λ,μ,n_prior,\nmax_mag,energy)','Spatial Features\n(dist_fault,\ndist_boundary)',
     'Temporal Features\n(hour_sin/cos,\ndoy_sin/cos)','Event Features\n(lat,lon,depth,\nmag,dt_prev)'],
    ['#1565C0','#2E7D32','#E65100','#6A1B9A']):
    box(ax,px,8.9,4.2,1.3,pc,pl,fs=9,rad=0.25)
for sx in [0.5,5.5,10.5,15.5]: arr(ax,sx+2.1,11.1,sx+2.1,10.2)
ax.text(11,8.35,'GRAPH CONSTRUCTION (3 Input Windows)',ha='center',va='center',
        fontsize=10,color='#555',style='italic')
for wx,wl,wc in zip([1.5,8.0,14.5],
    ['w24: Last 24h\nGraph\n(k-NN+ETAS edges)','w48: Last 48h\nGraph\n(k-NN+ETAS edges)',
     'w72: Last 72h\nGraph\n(k-NN+ETAS edges)'],['#1976D2','#388E3C','#F57C00']):
    box(ax,wx,6.9,5.0,1.1,wc,wl,fs=9,rad=0.25)
for wx in [1.5,8.0,14.5]: arr(ax,11,8.9,wx+2.5,8.0)
ax.text(11,6.4,'GATv2 ENCODER (shared weights, 3 layers)',ha='center',va='center',
        fontsize=10,color='#555',style='italic')
for wx,gc_ in zip([1.5,8.0,14.5],['#0D47A1','#1B5E20','#E65100']):
    box(ax,wx,5.3,5.0,0.85,gc_,'GATv2 × 3 layers','Mean+Max Pool → [B,256]',fs=8.5,rad=0.2)
for wx in [1.5,8.0,14.5]: arr(ax,wx+2.5,6.9,wx+2.5,6.15)
box(ax,7.5,4.2,7.0,0.85,'#37474F','Window Fusion: [e24|e48|e72] → Linear → [B,256]',fs=9,rad=0.25)
for wx in [1.5,8.0,14.5]: arr(ax,wx+2.5,5.3,11.0,5.05)
box(ax,7.5,2.8,7.0,0.85,'#4A148C','Transformer Encoder: 4 layers × 8 heads',
    'GNN token + 5 horizon tokens → [B,5,256]',fs=9,rad=0.25)
arr(ax,11.0,4.2,11.0,3.65)
ax.text(11,2.45,'PREDICTION HEADS (5 horizons × 3 tasks)',ha='center',va='center',
        fontsize=10,color='#555',style='italic')
for hx,hl,hc in zip([0.3,4.6,8.9,13.2,17.5],HORIZONS_PLOT,H_COLORS):
    box(ax,hx,1.55,3.8,0.75,hc,f'Horizon {hl}','occ | mag | lat,lon',fs=9,rad=0.2)
    arr(ax,11.0,2.8,hx+1.9,2.3)
for hx,hc in zip([0.3,4.6,8.9,13.2,17.5],H_COLORS):
    box(ax,hx,0.3,3.8,0.95,hc,'P(occ)  Mag  Lat,Lon',fs=8.5,rad=0.2)
    arr(ax,hx+1.9,1.55,hx+1.9,1.25)
ax.text(21.5,1.1,'Loss:\nBCE(occ)\n+0.3×MSE(mag)\n+0.3×MSE(loc)',
    ha='center',va='center',fontsize=8,color='#333',
    bbox=dict(boxstyle='round,pad=0.4',facecolor='#FFF9C4',edgecolor='#F9A825',linewidth=1.5))
plt.savefig(f"{FIG_DIR}fig01_methodology.png",dpi=180,bbox_inches='tight',facecolor='#FAFAFA')
plt.close(); print("  Saved fig01_methodology.png")

# fig02 — Training curves
fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.suptitle("AfterShockNet — Training History",fontsize=14,fontweight='bold')
epochs=list(range(1,len(history['train_loss'])+1))
best_ep=int(torch.tensor(history['val_loss']).argmin().item())+1
axes[0].plot(epochs,history['train_loss'],label='Train',linewidth=2.5,color='#1976D2')
axes[0].plot(epochs,history['val_loss'],label='Validation',linewidth=2.5,color='#D32F2F')
axes[0].axvline(best_ep,color='green',linestyle='--',linewidth=1.5,label=f'Best ep={best_ep}')
axes[0].set_xlabel('Epoch',fontsize=12); axes[0].set_ylabel('Normalised Loss',fontsize=12)
axes[0].set_title('Multi-Task Loss Convergence'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
for c,h in zip(H_COLORS,HORIZONS):
    axes[1].plot(epochs,[m[h]['f1'] for m in history['val_metrics']],
                 label=h,linewidth=2,color=c)
axes[1].set_xlabel('Epoch',fontsize=12); axes[1].set_ylabel('F1 Score',fontsize=12)
axes[1].set_title('Validation F1 per Horizon'); axes[1].legend()
axes[1].grid(True,alpha=0.3); axes[1].set_ylim(0,1.0)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig02_training_curves.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig02_training_curves.png")

# fig03 — ROC
fig,axes=plt.subplots(1,5,figsize=(25,5))
fig.suptitle("ROC Curves — All Models per Horizon",fontsize=14,fontweight='bold')
for ai,h in enumerate(HORIZONS):
    ax=axes[ai]; base=HORIZONS.index(h)*4
    occ_lbl=Y_test_t[:,base].float()
    for mn in MODEL_NAMES:
        if mn=='AfterShockNet': prob=asn_preds[h]['occ_prob']; av=asn_results[h]['auc']
        elif mn=='ETAS':        prob=to_ft(bl['etas_preds'][h]['occ_prob']); av=bl['etas_results'][h]['auc']
        elif mn=='XGBoost':     prob=to_ft(bl['xgb_preds'][h]['occ_prob']); av=bl['xgb_results'][h]['auc']
        elif mn=='LSTM':        prob=to_ft(bl['lstm_preds'][h]['occ_prob']); av=bl['lstm_results'][h]['auc']
        else:                   prob=to_ft(bl['tf_preds'][h]['occ_prob']);   av=bl['tf_results'][h]['auc']
        fprs,tprs=roc_curve_torch(occ_lbl,prob)
        ax.plot(fprs,tprs,color=COLORS_MODEL[mn],linewidth=2,label=f"{mn} ({av:.3f})")
    ax.plot([0,1],[0,1],'k--',alpha=0.4,linewidth=1)
    ax.set_title(f"Horizon {h}",fontsize=11,fontweight='bold')
    ax.set_xlabel('FPR',fontsize=10); ax.set_ylabel('TPR',fontsize=10)
    ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig03_roc_curves.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig03_roc_curves.png")

# fig04 — PR
fig,axes=plt.subplots(1,5,figsize=(25,5))
fig.suptitle("Precision-Recall Curves — All Models per Horizon",fontsize=14,fontweight='bold')
for ai,h in enumerate(HORIZONS):
    ax=axes[ai]; base=HORIZONS.index(h)*4
    occ_lbl=Y_test_t[:,base].float()
    for mn in MODEL_NAMES:
        if mn=='AfterShockNet': prob=asn_preds[h]['occ_prob']; pv=asn_results[h]['prauc']
        elif mn=='ETAS':        prob=to_ft(bl['etas_preds'][h]['occ_prob']); pv=bl['etas_results'][h].get('prauc',0)
        elif mn=='XGBoost':     prob=to_ft(bl['xgb_preds'][h]['occ_prob']); pv=bl['xgb_results'][h].get('prauc',0)
        elif mn=='LSTM':        prob=to_ft(bl['lstm_preds'][h]['occ_prob']); pv=bl['lstm_results'][h].get('prauc',0)
        else:                   prob=to_ft(bl['tf_preds'][h]['occ_prob']);   pv=bl['tf_results'][h].get('prauc',0)
        recs,precs=pr_curve_torch(occ_lbl,prob)
        ax.plot(recs,precs,color=COLORS_MODEL[mn],linewidth=2,label=f"{mn} ({pv:.3f})")
    ax.set_title(f"Horizon {h}",fontsize=11,fontweight='bold')
    ax.set_xlabel('Recall',fontsize=10); ax.set_ylabel('Precision',fontsize=10)
    ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig04_pr_curves.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig04_pr_curves.png")

# fig05 — Confusion matrices
fig,axes=plt.subplots(1,5,figsize=(25,5))
fig.suptitle("Confusion Matrices — AfterShockNet (Test Set)",fontsize=14,fontweight='bold')
for ax,h,c in zip(axes,HORIZONS,H_COLORS):
    r=asn_results[h]
    pred=(r['occ_prob']>=0.5).long(); lbl_l=r['occ_lbl'].long()
    cm=[[int(((lbl_l==t)&(pred==p)).sum()) for p in range(2)] for t in range(2)]
    ax.imshow(cm,cmap='Blues',vmin=0)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['No','Yes'],fontsize=11)
    ax.set_yticklabels(['No','Yes'],fontsize=11)
    ax.set_xlabel('Predicted',fontsize=10); ax.set_ylabel('True',fontsize=10)
    ax.set_title(f"{h}\nF1={r['f1']:.3f}  AUC={r['auc']:.3f}",fontsize=11,fontweight='bold')
    for ii in range(2):
        for jj in range(2):
            ax.text(jj,ii,f"{cm[ii][jj]:,}",ha='center',va='center',
                    fontsize=12,fontweight='bold',color='black')
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig05_confusion_matrices.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig05_confusion_matrices.png")

# figs 06-08 — bar charts
x_pos=list(range(len(HORIZONS))); bw=0.15
n_m=len(MODEL_NAMES); offsets=[((i-(n_m-1)/2)*bw) for i in range(n_m)]
for mk,title,ylabel,fname,ylim in [
    ('f1','F1 Score Comparison','F1 Score','fig06_f1_comparison.png',True),
    ('auc','AUC-ROC Comparison','AUC-ROC','fig07_auc_comparison.png',True),
    ('rmse','Magnitude RMSE Comparison','Mag RMSE','fig08_mag_rmse.png',False)]:
    fig,ax=plt.subplots(figsize=(14,6))
    fig.suptitle(f"{title} — All Models × All Horizons",fontsize=14,fontweight='bold')
    for mn,off in zip(MODEL_NAMES,offsets):
        vals=[get_metric(mn,h,mk) for h in HORIZONS]
        bars=ax.bar([p+off for p in x_pos],vals,bw,label=mn,
                    color=COLORS_MODEL[mn],edgecolor='black',linewidth=0.6)
        for bar,v in zip(bars,vals):
            if v>0.01:
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,
                        f'{v:.2f}',ha='center',va='bottom',fontsize=6.5,rotation=90)
    ax.set_xticks(x_pos); ax.set_xticklabels(HORIZONS,fontsize=12)
    if ylim: ax.set_ylim(0,1.15)
    ax.set_ylabel(ylabel,fontsize=12); ax.set_xlabel('Forecast Horizon',fontsize=12)
    ax.legend(fontsize=10,loc='upper left'); ax.grid(True,axis='y',alpha=0.3)
    if mk=='auc': ax.axhline(0.5,color='black',linestyle='--',alpha=0.4,linewidth=1)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}{fname}",dpi=150,bbox_inches='tight')
    plt.close(); print(f"  Saved {fname}")

# fig09 — Spatial error
fig,axes=plt.subplots(1,2,figsize=(14,6))
fig.suptitle("Spatial Location Error — AfterShockNet",fontsize=14,fontweight='bold')
for c,h in zip(H_COLORS,HORIZONS):
    r=asn_results[h]; mask=r['occ_lbl']==1
    if mask.sum()>0:
        hav=haversine_torch(r['lat_lbl'][mask],r['lon_lbl'][mask],
                            r['loc_pred'][mask,0],r['loc_pred'][mask,1])
        es_list=sorted(to_list(hav)); n_e=len(es_list)
        cdf=[float(i+1)/n_e for i in range(n_e)]
        axes[0].plot(es_list,cdf,color=c,linewidth=2,label=f"{h} (μ={r['mean_km']:.0f}km)")
        axes[1].hist(es_list,bins=50,alpha=0.5,color=c,label=h,density=True)
axes[0].set_xlabel('Haversine Error (km)',fontsize=12); axes[0].set_ylabel('CDF',fontsize=12)
axes[0].set_title('Spatial Error CDF'); axes[0].legend(fontsize=9)
axes[0].grid(True,alpha=0.3); axes[0].set_xlim(0,5000)
axes[1].set_xlabel('Haversine Error (km)',fontsize=12); axes[1].set_ylabel('Density',fontsize=12)
axes[1].set_title('Spatial Error Distribution'); axes[1].legend(fontsize=9)
axes[1].grid(True,alpha=0.3); axes[1].set_xlim(0,5000)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig09_spatial_error.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig09_spatial_error.png")

# fig10 — ETAS gain
fig,axes=plt.subplots(1,3,figsize=(18,6))
fig.suptitle("AfterShockNet Gain over ETAS Baseline",fontsize=14,fontweight='bold')
for ax,(mk,title) in zip(axes,[('f1','F1 Gain'),('auc','AUC Gain'),('rmse','Mag RMSE Reduction')]):
    gains=[]
    for h in HORIZONS:
        av=asn_results[h].get(mk,0); ev=bl['etas_results'][h].get(mk,0)
        gains.append(av-ev if mk!='rmse' else ev-av)
    colors_g=['#43A047' if g>=0 else '#E53935' for g in gains]
    bars=ax.bar(HORIZONS,gains,color=colors_g,edgecolor='black',linewidth=0.8)
    ax.axhline(0,color='black',linewidth=1)
    ax.set_title(title,fontsize=12,fontweight='bold'); ax.set_ylabel('Gain',fontsize=11)
    ax.grid(True,axis='y',alpha=0.3)
    for bar,v in zip(bars,gains):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+(0.002 if v>=0 else -0.005),
                f'{v:+.3f}',ha='center',va='bottom',fontsize=10,fontweight='bold')
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig10_etas_gain.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig10_etas_gain.png")

# fig11 — Probability distributions
fig,axes=plt.subplots(1,5,figsize=(25,4))
fig.suptitle("Predicted Probability Distributions — AfterShockNet",fontsize=14,fontweight='bold')
for ax,h in zip(axes,HORIZONS):
    r=asn_results[h]
    neg=to_list(r['occ_prob'][r['occ_lbl']==0])
    pos=to_list(r['occ_prob'][r['occ_lbl']==1])
    ax.hist(neg,bins=40,alpha=0.6,color='steelblue',label='True Neg',density=True)
    ax.hist(pos,bins=40,alpha=0.6,color='tomato',label='True Pos',density=True)
    ax.axvline(0.5,color='black',linestyle='--',linewidth=1.5)
    ax.set_title(f"{h}",fontsize=11,fontweight='bold')
    ax.set_xlabel('P(occurrence)',fontsize=10); ax.set_ylabel('Density',fontsize=10)
    ax.legend(fontsize=8); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig11_prob_distributions.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig11_prob_distributions.png")

# fig12 — Magnitude scatter
fig,axes=plt.subplots(1,5,figsize=(25,5))
fig.suptitle("Magnitude Prediction — AfterShockNet",fontsize=14,fontweight='bold')
for ax,h,c in zip(axes,HORIZONS,H_COLORS):
    r=asn_results[h]; mask=r['occ_lbl']==1; n=int(mask.sum().item())
    if n>0:
        ml=to_list(r['mag_lbl'][mask]); mp=to_list(r['mag_pred'][mask])
        ax.scatter(ml,mp,alpha=0.3,s=6,color=c)
        mn_v=min(min(ml),min(mp)); mx_v=max(max(ml),max(mp))
        ax.plot([mn_v,mx_v],[mn_v,mx_v],'r--',linewidth=1.5)
        ax.set_title(f"{h}\nRMSE={r['rmse']:.3f} n={n:,}",fontsize=10,fontweight='bold')
    ax.set_xlabel('True Mag',fontsize=10); ax.set_ylabel('Pred Mag',fontsize=10)
    ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig12_magnitude_scatter.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig12_magnitude_scatter.png")

# fig13 — Error analysis
fig,axes=plt.subplots(2,5,figsize=(25,10))
fig.suptitle("Error Analysis — False Positives & False Negatives",fontsize=14,fontweight='bold')
feat_t=to_ft(feat_df[FEATURE_COLS].values)
mag_col_f=FEATURE_COLS.index('mag')
etas_col_f=FEATURE_COLS.index('etas_lambda')
for col_i,h in enumerate(HORIZONS):
    r=asn_results[h]; pred=(r['occ_prob']>=0.5).float(); lbl=r['occ_lbl']
    fp_mask=((pred==1)&(lbl==0)); fn_mask=((pred==0)&(lbl==1)); tp_mask=((pred==1)&(lbl==1))
    ax=axes[0,col_i]
    for fmask,lbl_,col_ in [(fp_mask,'FP','orange'),(fn_mask,'FN','red'),(tp_mask,'TP','green')]:
        vals=to_list(feat_t[fmask,mag_col_f])
        if len(vals)>0: ax.hist(vals,bins=30,alpha=0.5,color=col_,label=lbl_,density=True)
    ax.set_title(f"{h} — Mag Dist",fontsize=10,fontweight='bold')
    ax.set_xlabel('Magnitude (scaled)',fontsize=9); ax.set_ylabel('Density',fontsize=9)
    ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
    ax=axes[1,col_i]
    for fmask,lbl_,col_ in [(fp_mask,'FP','orange'),(fn_mask,'FN','red')]:
        vals=to_list(feat_t[fmask,etas_col_f])
        if len(vals)>0: ax.hist(vals,bins=30,alpha=0.6,color=col_,label=lbl_,density=True)
    ax.set_title(f"{h} — ETAS λ",fontsize=10,fontweight='bold')
    ax.set_xlabel('ETAS λ (scaled)',fontsize=9); ax.set_ylabel('Density',fontsize=9)
    n_fp=int(fp_mask.sum()); n_fn=int(fn_mask.sum())
    ax.text(0.98,0.95,f'FP={n_fp:,}\nFN={n_fn:,}',transform=ax.transAxes,
            ha='right',va='top',fontsize=8,
            bbox=dict(boxstyle='round',facecolor='white',alpha=0.8))
    ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig13_error_analysis.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig13_error_analysis.png")

# fig14 — Global map
h_map='24h'; r=asn_results[h_map]; mask=r['occ_lbl']==1
try:
    import cartopy.crs as ccrs, cartopy.feature as cfeature
    fig=plt.figure(figsize=(18,9))
    ax=fig.add_subplot(1,1,1,projection=ccrs.Robinson())
    ax.set_global()
    ax.add_feature(cfeature.LAND,facecolor='#F5F5F5')
    ax.add_feature(cfeature.OCEAN,facecolor='#BBDEFB')
    ax.add_feature(cfeature.COASTLINE,linewidth=0.5,edgecolor='#555')
    ax.add_feature(cfeature.BORDERS,linewidth=0.3,linestyle=':')
    ax.gridlines(draw_labels=False,linewidth=0.3,color='gray',alpha=0.4)
    if mask.sum()>0:
        ax.scatter(to_list(r['lon_lbl'][mask]),to_list(r['lat_lbl'][mask]),
                   c='red',s=10,alpha=0.6,transform=ccrs.PlateCarree(),
                   label='True aftershock',zorder=5)
        ax.scatter(to_list(r['loc_pred'][mask,1]),to_list(r['loc_pred'][mask,0]),
                   c='blue',s=10,alpha=0.4,transform=ccrs.PlateCarree(),
                   label='Predicted location',zorder=4)
    ax.set_title(f"Global Forecast Map — {h_map}\nRed=True  Blue=Predicted",
                 fontsize=13,fontweight='bold')
    ax.legend(loc='lower left',fontsize=9)
except ImportError:
    fig,ax=plt.subplots(figsize=(18,9))
    ax.set_facecolor('#BBDEFB')
    if mask.sum()>0:
        ax.scatter(to_list(r['lon_lbl'][mask]),to_list(r['lat_lbl'][mask]),
                   c='red',s=8,alpha=0.5,label='True aftershock')
        ax.scatter(to_list(r['loc_pred'][mask,1]),to_list(r['loc_pred'][mask,0]),
                   c='blue',s=8,alpha=0.4,label='Predicted location')
    ax.set_xlim(-180,180); ax.set_ylim(-90,90)
    ax.set_xlabel('Longitude',fontsize=12); ax.set_ylabel('Latitude',fontsize=12)
    ax.set_title(f"Global Forecast Map — {h_map}",fontsize=13,fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig14_global_map.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig14_global_map.png")

# fig15 — Heatmap
fig,axes=plt.subplots(1,3,figsize=(18,7))
fig.suptitle("Performance Heatmap — All Models × All Horizons",fontsize=14,fontweight='bold')
for ax,mk,title in zip(axes,['f1','auc','rmse'],['F1 Score','AUC-ROC','Mag RMSE']):
    mat=[[get_metric(mn,h,mk) for h in HORIZONS] for mn in MODEL_NAMES]
    cmap='RdYlGn' if mk!='rmse' else 'RdYlGn_r'
    im=ax.imshow(mat,cmap=cmap,aspect='auto')
    ax.set_xticks(list(range(len(HORIZONS)))); ax.set_xticklabels(HORIZONS,fontsize=11)
    ax.set_yticks(list(range(len(MODEL_NAMES)))); ax.set_yticklabels(MODEL_NAMES,fontsize=11)
    ax.set_title(title,fontsize=12,fontweight='bold'); plt.colorbar(im,ax=ax,shrink=0.8)
    for i in range(len(MODEL_NAMES)):
        for j in range(len(HORIZONS)):
            ax.text(j,i,f'{mat[i][j]:.3f}',ha='center',va='center',
                    fontsize=9,fontweight='bold',color='black')
plt.tight_layout()
plt.savefig(f"{FIG_DIR}fig15_heatmap.png",dpi=150,bbox_inches='tight')
plt.close(); print("  Saved fig15_heatmap.png")

# ── Final summary ─────────────────────────────────────────────
print(f"\n{'='*65}")
print("Step 6 + Q1 Analyses Complete")
print(f"{'='*65}")
print("\nOriginal figures:")
for f in ["fig01_methodology.png","fig02_training_curves.png",
          "fig03_roc_curves.png","fig04_pr_curves.png",
          "fig05_confusion_matrices.png","fig06_f1_comparison.png",
          "fig07_auc_comparison.png","fig08_mag_rmse.png",
          "fig09_spatial_error.png","fig10_etas_gain.png",
          "fig11_prob_distributions.png","fig12_magnitude_scatter.png",
          "fig13_error_analysis.png","fig14_global_map.png","fig15_heatmap.png"]:
    print(f"  {f}")
print("\nNew Q1 figures:")
for f in ["fig16_significance_heatmap.png  — Bootstrap + Wilcoxon p-values",
          "fig17_ablation_study.png         — Ablation F1/PR-AUC/RMSE",
          "fig18_ablation_delta.png         — Component drop heatmap",
          "fig19_tectonic_zone_performance.png — Zone-stratified bars",
          "fig20_tectonic_f1_heatmap.png    — Zone × Horizon F1 heatmap",
          "fig21_m6_sequence_performance.png   — M>=6 F1 and PR-AUC",
          "fig22_m6_vs_all_comparison.png   — M>=6 vs all-events",
          "fig23_m6_magnitude_dist.png      — M>=6 anchor distribution"]:
    print(f"  {f}")
print("\nCSV outputs:")
for f in ["evaluation_results.csv","statistical_significance.csv",
          "wilcoxon_results.csv","ablation_results.csv",
          "tectonic_zone_results.csv","m6_sequence_results.csv"]:
    print(f"  {OUT_DIR}{f}")
print(f"\n  AfterShockNet Test Summary:")
print(f"  {'Horizon':<8} {'F1':>7} {'AUC':>7} {'PR-AUC':>8} {'MagRMSE':>9} {'SpatMean':>10}")
print(f"  {'-'*52}")
for h in HORIZONS:
    r=asn_results[h]
    print(f"  {h:<8} {r['f1']:>7.4f} {r['auc']:>7.4f} "
          f"{r['prauc']:>8.4f} {r['rmse']:>9.4f} {r['mean_km']:>9.1f}km")